In [20]:
folder_of_data = "/data001/projects/ising/energy_decomposition"

In [21]:
!pip3 install matplotlib
!pip3 install pandas
!pip3 install h5py
!pip3 install scipy numba cython multiprocess boost==1.74
# !pip3 install coniii


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable
  Using cached numba-0.60.0-cp39-cp39-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.7 kB)
  Using cached cython-3.2.3-cp39-cp39-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (4.5 kB)
  Using cached multiprocess-0.70.18-py39-none-any.whl.metadata (7.5 kB)
ERROR: Could not find a version that satisfies the requ

In [22]:
import matplotlib.pyplot as plt
import pandas as pd
import os

In [23]:
def find_file_recursive(root_folder, file_name, current_depth=0, max_depth=None, verbose=True):
    """
    Recursively search for a file in all subdirectories
    
    Args:
        root_folder (str): Path to start searching from
        file_name (str): Name of the file to look for
        current_depth (int): Current recursion depth (for display purposes)
        max_depth (int): Maximum depth to search (None for unlimited)
        verbose (bool): Whether to print detailed output
    
    Returns:
        list: List of paths where the file was found
    """
    found_paths = []
    indent = "  " * current_depth
    
    # Check if we've reached max depth
    if max_depth is not None and current_depth > max_depth:
        return found_paths
    
    # Normalize the path to handle different path separators
    root_folder = os.path.normpath(root_folder)
    
    # Check if the folder exists and is accessible
    if not os.path.exists(root_folder):
        if verbose:
            print(f"{indent}❌ Folder does not exist: {root_folder}")
        return found_paths
    
    if not os.path.isdir(root_folder):
        if verbose:
            print(f"{indent}❌ Not a directory: {root_folder}")
        return found_paths
    
    try:
        items = os.listdir(root_folder)
    except PermissionError:
        if verbose:
            print(f"{indent}❌ Permission denied: {root_folder}")
        return found_paths
    except Exception as e:
        if verbose:
            print(f"{indent}❌ Error accessing {root_folder}: {e}")
        return found_paths
    
    if verbose:
        print(f"{indent}📁 Searching in: {root_folder}")
    
    # Check if the file exists in current directory
    file_path = os.path.join(root_folder, file_name)
    if os.path.isfile(file_path):  # Use isfile instead of exists to ensure it's a file
        if verbose:
            print(f"{indent}  ✅ Found '{file_name}'")
        found_paths.append(file_path)
    else:
        if verbose:
            print(f"{indent}  ❌ '{file_name}' not found")
    
    # Recursively search in subdirectories
    subdirs = []
    for item in items:
        item_path = os.path.join(root_folder, item)
        if os.path.isdir(item_path):
            subdirs.append(item_path)
    
    if verbose and subdirs:
        print(f"{indent}  📂 Found {len(subdirs)} subdirectories")
    
    for subdir in subdirs:
        try:
            subfolder_results = find_file_recursive(subdir, file_name, current_depth + 1, max_depth, verbose)
            found_paths.extend(subfolder_results)
        except Exception as e:
            if verbose:
                print(f"{indent}  ❌ Error processing {subdir}: {e}")
    
    return found_paths

In [24]:
def extract_results_folder(file_path):
    """
    Extract the folder name that ends with '_results' from a file path
    
    Args:
        file_path (str): Full path to a file
        
    Returns:
        str: The folder name ending with '_results', or None if not found
    """
    # Split the path into components
    path_parts = file_path.split(os.sep)
    
    # Look for a folder that ends with '_results'
    for part in path_parts:
        if part.endswith('_results'):
            return part
    
    return None

In [25]:
def ensure_directory_exists(directory_path):
    """
    Check if a directory exists and create it (including all parent directories) if it doesn't.
    
    Args:
        directory_path (str): Path to the directory to check/create
    
    Returns:
        bool: True if directory exists or was created successfully, False if creation failed
    """
    try:
        os.makedirs(directory_path, exist_ok=True)
        return True
    except Exception as e:
        print(f"Error creating directory '{directory_path}': {e}")
        return False

In [26]:
# review files of path saved in folder_of_data variables
# not all mice have full data, still working on incomplete mice
# This will help you select which csv in the path to use
all_reach_states = find_file_recursive(folder_of_data, "per_reach_state.csv")

📁 Searching in: /data001/projects/ising/energy_decomposition
  ❌ 'per_reach_state.csv' not found
  📂 Found 23 subdirectories
  📁 Searching in: /data001/projects/ising/energy_decomposition/210421_results
    ❌ 'per_reach_state.csv' not found
    📂 Found 1 subdirectories
    📁 Searching in: /data001/projects/ising/energy_decomposition/210421_results/210421_rep1
      ❌ 'per_reach_state.csv' not found
      📂 Found 4 subdirectories
      📁 Searching in: /data001/projects/ising/energy_decomposition/210421_results/210421_rep1/begin_reach
        ❌ 'per_reach_state.csv' not found
      📁 Searching in: /data001/projects/ising/energy_decomposition/210421_results/210421_rep1/mid_reach
        ❌ 'per_reach_state.csv' not found
      📁 Searching in: /data001/projects/ising/energy_decomposition/210421_results/210421_rep1/post_reach
        ❌ 'per_reach_state.csv' not found
      📁 Searching in: /data001/projects/ising/energy_decomposition/210421_results/210421_rep1/full_reach
        ❌ 'per_reach_

In [27]:
print("Columns: ")
import numpy as np
print("- "+"\n- ".join(pd.read_csv(all_reach_states[0]).columns))
print(all_reach_states[5])
reach = pd.read_csv(all_reach_states[3])

reach[np.logical_and(reach['reach_idx'] == 0, reach['stim'] == 0)]

Columns: 
- reach_idx
- stim
- x
- y
- z
- firing_rate
- energy
- j
- h
/data001/projects/ising/energy_decomposition/210425_results/210425_rep1/mid_reach/per_reach_state.csv


,reach_idx,stim,x,y,z,firing_rate,energy,j,h
0,0,0,0.026840,0.118307,0.811070,0.133333,-0.044844,0.00000,0.044844
1,0,0,0.026830,0.118189,0.811066,0.142857,-0.049972,0.00000,0.049972
2,0,0,0.026794,0.118156,0.811135,0.125000,0.206248,0.00000,-0.206248
3,0,0,0.026739,0.118220,0.811268,0.133333,-0.000000,0.00000,0.000000
4,0,0,0.026693,0.118371,0.811452,0.120000,0.206248,0.00000,-0.206248
...,...,...,...,...,...,...,...,...,...
695,0,0,0.098536,0.062668,0.410207,0.120000,-0.158886,0.06407,0.094816
696,0,0,0.097551,0.063265,0.411807,0.111111,-0.000000,0.00000,0.000000
697,0,0,0.096537,0.063903,0.413030,0.125000,-0.000000,0.00000,0.000000
698,0,0,0.095601,0.064542,0.413826,0.114286,-0.000000,0.00000,0.000000


In [37]:
all_reach_states[0].split("_results")[0][-6:]

'210423'

In [29]:
def calculate_statistics_from_dataframe(df, confidence=0.8, output_dir=None):
    """
    Calculate statistics across reaches for kinematic (x, y, z) and neural data from a dataframe.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with columns: reach_idx, stim, x, y, z, firing_rate, energy, j, h
        Each row represents a single time point for a specific reach in a specific stimulus
    confidence : float, optional
        Confidence level for intervals (default=0.8)
    output_dir : str, optional
        Directory to save statistics results
        
    Returns:
    --------
    dict
        Dictionary containing mean and confidence intervals for each measure
    """
    from energy_over_time_script.utils import mean_confidence_interval
    
    # Get unique stimuli
    stimuli = sorted(df['stim'].unique())
    
    # Initialize containers for results
    x_kinematics_stats = []
    y_kinematics_stats = []
    z_kinematics_stats = []
    energy_stats = []
    firing_rate_stats = []
    j_values_stats = []
    h_values_stats = []
    
    # Calculate statistics for each stimulation condition
    for stim in stimuli:
        # Filter data for this stimulus
        stim_data = df[df['stim'] == stim].copy()
        
        # Group by time index (assuming rows are ordered by time)
        # We need to identify unique time points across all reaches
        stim_data['time_idx'] = stim_data.groupby('reach_idx').cumcount()
        
        # Get the maximum time length
        max_time = stim_data['time_idx'].max() + 1
        
        # Initialize lists for this stimulus
        x_mean, x_lower, x_upper = [], [], []
        y_mean, y_lower, y_upper = [], [], []
        z_mean, z_lower, z_upper = [], [], []
        energy_mean, energy_lower, energy_upper = [], [], []
        firing_mean, firing_lower, firing_upper = [], [], []
        j_mean, j_lower, j_upper = [], [], []
        h_mean, h_lower, h_upper = [], [], []
        
        # Calculate statistics for each time point
        for t in range(max_time):
            time_data = stim_data[stim_data['time_idx'] == t]
            
            if len(time_data) > 0:
                # X-coordinate statistics
                x_m, x_ml, x_mu = mean_confidence_interval(time_data['x'].values, confidence)
                x_mean.append(x_m)
                x_lower.append(x_ml)
                x_upper.append(x_mu)
                
                # Y-coordinate statistics
                y_m, y_ml, y_mu = mean_confidence_interval(time_data['y'].values, confidence)
                y_mean.append(y_m)
                y_lower.append(y_ml)
                y_upper.append(y_mu)
                
                # Z-coordinate statistics
                z_m, z_ml, z_mu = mean_confidence_interval(time_data['z'].values, confidence)
                z_mean.append(z_m)
                z_lower.append(z_ml)
                z_upper.append(z_mu)
                
                # Energy statistics
                e_m, e_ml, e_mu = mean_confidence_interval(time_data['energy'].values, confidence)
                energy_mean.append(e_m)
                energy_lower.append(e_ml)
                energy_upper.append(e_mu)
                
                # Firing rate statistics
                f_m, f_ml, f_mu = mean_confidence_interval(time_data['firing_rate'].values, confidence)
                firing_mean.append(f_m)
                firing_lower.append(f_ml)
                firing_upper.append(f_mu)
                
                # J statistics
                j_m, j_ml, j_mu = mean_confidence_interval(time_data['j'].values, confidence)
                j_mean.append(j_m)
                j_lower.append(j_ml)
                j_upper.append(j_mu)
                
                # H statistics
                h_m, h_ml, h_mu = mean_confidence_interval(time_data['h'].values, confidence)
                h_mean.append(h_m)
                h_lower.append(h_ml)
                h_upper.append(h_mu)
        
        # Store statistics for this stimulus
        x_kinematics_stats.append({
            'mean': x_mean,
            'lower': x_lower,
            'upper': x_upper
        })
        
        y_kinematics_stats.append({
            'mean': y_mean,
            'lower': y_lower,
            'upper': y_upper
        })
        
        z_kinematics_stats.append({
            'mean': z_mean,
            'lower': z_lower,
            'upper': z_upper
        })
        
        energy_stats.append({
            'mean': energy_mean,
            'lower': energy_lower,
            'upper': energy_upper
        })
        
        firing_rate_stats.append({
            'mean': firing_mean,
            'lower': firing_lower,
            'upper': firing_upper
        })
        
        j_values_stats.append({
            'mean': j_mean,
            'lower': j_lower,
            'upper': j_upper
        })
        
        h_values_stats.append({
            'mean': h_mean,
            'lower': h_lower,
            'upper': h_upper
        })
        
        # Save statistics to CSV and create plots if output directory is provided
        if output_dir:
            os.makedirs(output_dir, exist_ok=True)
            time_points = list(range(len(x_mean)))
            
            # Save X-coordinate statistics
            x_stats_df = pd.DataFrame({
                'Time': time_points,
                'Mean': x_mean,
                'Lower_CI': x_lower,
                'Upper_CI': x_upper
            })
            x_stats_df.to_csv(os.path.join(output_dir, f"x_kinematics_stats_stim_{stim}.csv"), index=False)
            
            # Save Y-coordinate statistics
            y_stats_df = pd.DataFrame({
                'Time': time_points,
                'Mean': y_mean,
                'Lower_CI': y_lower,
                'Upper_CI': y_upper
            })
            y_stats_df.to_csv(os.path.join(output_dir, f"y_kinematics_stats_stim_{stim}.csv"), index=False)
            
            # Save Z-coordinate statistics
            z_stats_df = pd.DataFrame({
                'Time': time_points,
                'Mean': z_mean,
                'Lower_CI': z_lower,
                'Upper_CI': z_upper
            })
            z_stats_df.to_csv(os.path.join(output_dir, f"z_kinematics_stats_stim_{stim}.csv"), index=False)
            
            # Save Energy statistics
            energy_stats_df = pd.DataFrame({
                'Time': time_points,
                'Mean': energy_mean,
                'Lower_CI': energy_lower,
                'Upper_CI': energy_upper
            })
            energy_stats_df.to_csv(os.path.join(output_dir, f"energy_stats_stim_{stim}.csv"), index=False)
            
            # Save Firing Rate statistics
            firing_stats_df = pd.DataFrame({
                'Time': time_points,
                'Mean': firing_mean,
                'Lower_CI': firing_lower,
                'Upper_CI': firing_upper
            })
            firing_stats_df.to_csv(os.path.join(output_dir, f"firing_rate_stats_stim_{stim}.csv"), index=False)
            
            # Save J statistics
            j_stats_df = pd.DataFrame({
                'Time': time_points,
                'Mean': j_mean,
                'Lower_CI': j_lower,
                'Upper_CI': j_upper
            })
            j_stats_df.to_csv(os.path.join(output_dir, f"j_stats_stim_{stim}.csv"), index=False)
            
            # Save H statistics
            h_stats_df = pd.DataFrame({
                'Time': time_points,
                'Mean': h_mean,
                'Lower_CI': h_lower,
                'Upper_CI': h_upper
            })
            h_stats_df.to_csv(os.path.join(output_dir, f"h_stats_stim_{stim}.csv"), index=False)
            
            # Create stacked plots for X, Y, Z coordinates, Energy, and Firing Rate
            fig, axs = plt.subplots(5, 1, figsize=(12, 20), sharex=True)
            
            # X-coordinate plot
            axs[0].plot(time_points, x_mean, 'b-', linewidth=2, label='Mean X')
            axs[0].fill_between(time_points, x_lower, x_upper, alpha=0.3, color='blue', label=f'{int(confidence*100)}% CI')
            axs[0].set_ylabel('X Coordinate')
            axs[0].set_title(f'X-Coordinate Statistics - Stimulus {stim}')
            axs[0].legend()
            axs[0].grid(True, alpha=0.3)
            
            # Y-coordinate plot
            axs[1].plot(time_points, y_mean, 'g-', linewidth=2, label='Mean Y')
            axs[1].fill_between(time_points, y_lower, y_upper, alpha=0.3, color='green', label=f'{int(confidence*100)}% CI')
            axs[1].set_ylabel('Y Coordinate')
            axs[1].set_title(f'Y-Coordinate Statistics - Stimulus {stim}')
            axs[1].legend()
            axs[1].grid(True, alpha=0.3)
            
            # Z-coordinate plot
            axs[2].plot(time_points, z_mean, 'r-', linewidth=2, label='Mean Z')
            axs[2].fill_between(time_points, z_lower, z_upper, alpha=0.3, color='red', label=f'{int(confidence*100)}% CI')
            axs[2].set_ylabel('Z Coordinate')
            axs[2].set_title(f'Z-Coordinate Statistics - Stimulus {stim}')
            axs[2].legend()
            axs[2].grid(True, alpha=0.3)
            
            # Energy plot
            axs[3].plot(time_points, energy_mean, 'purple', linewidth=2, label='Mean Energy')
            axs[3].fill_between(time_points, energy_lower, energy_upper, alpha=0.3, color='purple', label=f'{int(confidence*100)}% CI')
            axs[3].set_ylabel('Energy')
            axs[3].set_title(f'Energy Statistics - Stimulus {stim}')
            axs[3].legend()
            axs[3].grid(True, alpha=0.3)
            
            # Firing Rate plot
            axs[4].plot(time_points, firing_mean, 'orange', linewidth=2, label='Mean Firing Rate')
            axs[4].fill_between(time_points, firing_lower, firing_upper, alpha=0.3, color='orange', label=f'{int(confidence*100)}% CI')
            axs[4].set_ylabel('Firing Rate')
            axs[4].set_xlabel('Time Points')
            axs[4].set_title(f'Firing Rate Statistics - Stimulus {stim}')
            axs[4].legend()
            axs[4].grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, f"stacked_statistics_stim_{stim}.png"), dpi=300, bbox_inches='tight')
            plt.close()
    
    return {
        'x_kinematics': x_kinematics_stats,
        'y_kinematics': y_kinematics_stats,
        'z_kinematics': z_kinematics_stats,
        'energy': energy_stats,
        'firing_rate': firing_rate_stats,
        'j_values': j_values_stats,
        'h_values': h_values_stats
    }


def plot_energy_across_time_from_stats(stats, critical_energy, output_dir, title_prefix="", show_mid_point=True, window_size=10, temperature=1.0):
    """
    Plot energy and kinematic data (x, y, z coordinates) across time from pre-computed statistics.
    
    Parameters:
    -----------
    stats : dict
        Statistics dictionary containing x_kinematics, y_kinematics, z_kinematics, energy, firing_rate, j_values, h_values
    critical_energy : float
        Critical energy from phase transition analysis
    output_dir : str
        Directory to save plots
    title_prefix : str, optional
        Prefix for plot titles (default="")
    show_mid_point : bool, optional
        Whether to mark the middle point (default=True)
    window_size : int, optional
        Size of the sliding window for firing rate calculation (default=10)
    temperature : float, optional
        Temperature parameter for Boltzmann probability calculation (default=1.0)
        
    Returns:
    --------
    None
    """
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import os
    
    os.makedirs(output_dir, exist_ok=True)
    
    x_kinematics = stats['x_kinematics']
    y_kinematics = stats['y_kinematics']
    z_kinematics = stats['z_kinematics']
    energy = stats['energy']
    firing_rate = stats.get('firing_rate', None)
    j_values = stats.get('j_values', None)
    h_values = stats.get('h_values', None)
    
    # Debug: print what we have
    print(f"j_values is None: {j_values is None}")
    print(f"h_values is None: {h_values is None}")
    if j_values is not None:
        print(f"j_values length: {len(j_values)}")
    if h_values is not None:
        print(f"h_values length: {len(h_values)}")
    
    # Create overlapped firing rate plot (all stimuli together)
    if firing_rate is not None:
        plt.figure(figsize=(14, 6))
        
        # Updated color scheme: red, blue, purple for first 3 stimuli
        colors = ['#FF0000', '#0000FF', '#9900FF', '#FF6B6B', '#4ECDC4', '#45B7D1']
        
        for i, firing_data in enumerate(firing_rate):
            color = colors[i % len(colors)]
            plt.plot(firing_data['mean'], '-', linewidth=2, 
                    label=f'Stim_{i} Firing Rate', color=color, alpha=0.8)
            plt.fill_between(range(len(firing_data['mean'])), 
                           firing_data['lower'],
                           firing_data['upper'],
                           color=color, alpha=0.15)
        
        # Mark midpoint if requested
        if show_mid_point and len(firing_rate[0]['mean']) > 100:
            mid_point = len(firing_rate[0]['mean']) // 2
            plt.axvline(x=mid_point, color='g', linestyle='--', alpha=0.7, label='midpoint')
        
        plt.xlabel("Time")
        plt.ylabel("Firing Rate")
        plt.title(f"{title_prefix} Firing Rate Comparison Across All Stimuli")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f"firing_rate_all_stimuli.png"), dpi=300, bbox_inches='tight')
        plt.close()
        
        # Save combined firing rate data to CSV
        combined_firing_df = pd.DataFrame({'Time': range(len(firing_rate[0]['mean']))})
        for i, firing_data in enumerate(firing_rate):
            combined_firing_df[f'Stim_{i}_Mean'] = firing_data['mean']
            combined_firing_df[f'Stim_{i}_Upper_CI'] = firing_data['upper']
            combined_firing_df[f'Stim_{i}_Lower_CI'] = firing_data['lower']
        combined_firing_df.to_csv(os.path.join(output_dir, f"firing_rate_all_stimuli.csv"), index=False)
    
    # Plot individual stimulus data
    for i, (x_kin, y_kin, z_kin, eng) in enumerate(zip(x_kinematics, y_kinematics, z_kinematics, energy)):
        # Determine layout: we'll use 2 columns for stim 1 and 2
        # Left column: X, Y, Z, Energy, Boltzmann
        # Right column: J values (stim 1), H values (stim 2), empty otherwise
        
        n_rows = 5  # X, Y, Z, Energy, Boltzmann
        
        # Check if we need the right column for this stimulus
        has_right_column = False
        if i == 1 and j_values is not None and len(j_values) > i:
            has_right_column = True
            print(f"Stim {i}: Will show J values in right column")
        elif i == 2 and h_values is not None and len(h_values) > i:
            has_right_column = True
            print(f"Stim {i}: Will show H values in right column")
        else:
            print(f"Stim {i}: No right column")
        
        if has_right_column:
            fig = plt.figure(figsize=(20, 3*n_rows))
            gs = fig.add_gridspec(n_rows, 2, hspace=0.3, wspace=0.3)
        else:
            fig = plt.figure(figsize=(12, 3*n_rows))
            gs = fig.add_gridspec(n_rows, 1, hspace=0.3)
        
        # X-coordinate subplot (left column, row 0)
        ax = fig.add_subplot(gs[0, 0])
        ax.set_title(f"{title_prefix} X-Coordinate Over Time, Stim_{i}")
        
        ax.plot(x_kin['mean'], '-r', linewidth=2, label='mean X')
        ax.plot(x_kin['upper'], '-b', label='upper CI', alpha=0.5)
        ax.plot(x_kin['lower'], '-b', label='lower CI', alpha=0.5)
        
        # Plot other stimulus means for comparison if available
        for j, other_x_kin in enumerate(x_kinematics):
            if j != i:
                ax.plot(other_x_kin['mean'], '-', label=f"mean X stim_{j}", alpha=0.7)
        
        # Fill between confidence intervals
        ax.fill_between(list(range(len(x_kin['mean']))), x_kin['upper'], x_kin['lower'], 
                         color="blue", alpha=0.15)
        
        # Mark midpoint if requested
        if show_mid_point and len(x_kin['mean']) > 100:
            mid_point = len(x_kin['mean']) // 2
            ax.axvline(x=mid_point, color='g', linestyle='--', alpha=0.7, label='midpoint')
        
        ax.set_ylabel("X Position")
        ax.legend()
        ax.grid(alpha=0.3)
        
        # Y-coordinate subplot (left column, row 1)
        ax = fig.add_subplot(gs[1, 0])
        ax.set_title(f"{title_prefix} Y-Coordinate Over Time, Stim_{i}")
        
        ax.plot(y_kin['mean'], '-g', linewidth=2, label='mean Y')
        ax.plot(y_kin['upper'], '-b', label='upper CI', alpha=0.5)
        ax.plot(y_kin['lower'], '-b', label='lower CI', alpha=0.5)
        
        # Plot other stimulus means for comparison if available
        for j, other_y_kin in enumerate(y_kinematics):
            if j != i:
                ax.plot(other_y_kin['mean'], '-', label=f"mean Y stim_{j}", alpha=0.7)
        
        # Fill between confidence intervals
        ax.fill_between(list(range(len(y_kin['mean']))), y_kin['upper'], y_kin['lower'], 
                         color="green", alpha=0.15)
        
        # Mark midpoint if requested
        if show_mid_point and len(y_kin['mean']) > 100:
            mid_point = len(y_kin['mean']) // 2
            ax.axvline(x=mid_point, color='g', linestyle='--', alpha=0.7, label='midpoint')
        
        ax.set_ylabel("Y Position")
        ax.legend()
        ax.grid(alpha=0.3)
        
        # Z-coordinate subplot (left column, row 2)
        ax = fig.add_subplot(gs[2, 0])
        ax.set_title(f"{title_prefix} Z-Coordinate Over Time, Stim_{i}")
        
        ax.plot(z_kin['mean'], '-r', linewidth=2, label='mean Z', color='red')
        ax.plot(z_kin['upper'], '-b', label='upper CI', alpha=0.5)
        ax.plot(z_kin['lower'], '-b', label='lower CI', alpha=0.5)
        
        # Plot other stimulus means for comparison if available
        for j, other_z_kin in enumerate(z_kinematics):
            if j != i:
                ax.plot(other_z_kin['mean'], '-', label=f"mean Z stim_{j}", alpha=0.7)
        
        # Fill between confidence intervals
        ax.fill_between(list(range(len(z_kin['mean']))), z_kin['upper'], z_kin['lower'], 
                         color="red", alpha=0.15)
        
        # Mark midpoint if requested
        if show_mid_point and len(z_kin['mean']) > 100:
            mid_point = len(z_kin['mean']) // 2
            ax.axvline(x=mid_point, color='g', linestyle='--', alpha=0.7, label='midpoint')
        
        ax.set_ylabel("Z Position")
        ax.legend()
        ax.grid(alpha=0.3)
        
        # Energy subplot (left column, row 3)
        ax = fig.add_subplot(gs[3, 0])
        ax.set_title(f"{title_prefix} Energy of Neural Activity Over Time, Stim_{i}")
        
        # Fill between confidence intervals
        ax.fill_between(list(range(len(eng['mean']))), eng['upper'], eng['lower'], 
                         color="purple", alpha=0.15)
        
        ax.plot(eng['mean'], '-', linewidth=2, label='mean Energy', color='purple')
        
        # Plot other stimulus means for comparison if available
        for j, other_eng in enumerate(energy):
            if j != i:
                ax.plot(other_eng['mean'], '-', label=f"mean Energy stim_{j}", alpha=0.7)
        
        # Mark critical energy and mean energy
        ax.axhline(y=critical_energy, color='r', linestyle='--',
                   label=f"Critical Energy = {critical_energy:.2f}")
        ax.axhline(y=np.mean(eng['mean']), color='b', linestyle='-.',
                   label=f"Mean Energy = {np.mean(eng['mean']):.2f}")
        
        # Mark midpoint if requested
        if show_mid_point and len(eng['mean']) > 100:
            mid_point = len(eng['mean']) // 2
            ax.axvline(x=mid_point, color='g', linestyle='--', alpha=0.7, label='midpoint')
        
        ax.set_ylabel("Energy")
        ax.legend()
        ax.grid(alpha=0.3)
        
        # Boltzmann Probability subplot (left column, row 4)
        ax = fig.add_subplot(gs[4, 0])
        ax.set_title(f"{title_prefix} Boltzmann Probability Over Time, Stim_{i}")
        
        # Calculate Boltzmann probabilities: exp(-Energy/T)
        boltzmann_mean = np.exp(-np.array(eng['mean']) / temperature)
        boltzmann_upper = np.exp(-np.array(eng['lower']) / temperature)  # Note: lower energy -> higher probability
        boltzmann_lower = np.exp(-np.array(eng['upper']) / temperature)  # Note: higher energy -> lower probability
        
        # Fill between confidence intervals
        ax.fill_between(list(range(len(boltzmann_mean))), boltzmann_upper, boltzmann_lower, 
                         color="teal", alpha=0.15)
        
        ax.plot(boltzmann_mean, '-', linewidth=2, label='mean Boltzmann Prob', color='teal')
        
        # Plot other stimulus Boltzmann probabilities for comparison if available
        for j, other_eng in enumerate(energy):
            if j != i:
                other_boltzmann = np.exp(-np.array(other_eng['mean']) / temperature)
                ax.plot(other_boltzmann, '-', label=f"Boltzmann Prob stim_{j}", alpha=0.7)
        
        # Mark midpoint if requested
        if show_mid_point and len(boltzmann_mean) > 100:
            mid_point = len(boltzmann_mean) // 2
            ax.axvline(x=mid_point, color='g', linestyle='--', alpha=0.7, label='midpoint')
        
        ax.set_xlabel("Time")
        ax.set_ylabel("Boltzmann Probability")
        ax.legend()
        ax.grid(alpha=0.3)
        
        # Right column plots
        if has_right_column:
            if i == 1 and j_values is not None and len(j_values) > i:
                # J values plot for stimulus 1 (right column, spanning rows 0-1)
                ax = fig.add_subplot(gs[0:2, 1])
                ax.set_title(f"{title_prefix} J Values (Local Interactions), Stim_{i}")
                
                ax.plot(j_values[i]['mean'], '-', linewidth=3, label=f"mean J", color='darkblue')
                ax.fill_between(range(len(j_values[i]['mean'])), 
                               j_values[i]['lower'],
                               j_values[i]['upper'],
                               color='darkblue', alpha=0.15)
                
                if show_mid_point and len(j_values[i]['mean']) > 100:
                    mid_point = len(j_values[i]['mean']) // 2
                    ax.axvline(x=mid_point, color='g', linestyle='--', alpha=0.7, label='midpoint')
                
                ax.set_ylabel("J Value", fontsize=12)
                ax.set_xlabel("Time", fontsize=12)
                ax.legend()
                ax.grid(alpha=0.3)
                print(f"Added J values plot for stim {i}")
                
            elif i == 2 and h_values is not None and len(h_values) > i:
                # H values plot for stimulus 2 (right column, spanning rows 0-1)
                ax = fig.add_subplot(gs[0:2, 1])
                ax.set_title(f"{title_prefix} H Values (Local Fields), Stim_{i}")
                
                ax.plot(h_values[i]['mean'], '-', linewidth=3, label=f"mean H", color='darkgreen')
                ax.fill_between(range(len(h_values[i]['mean'])), 
                               h_values[i]['lower'],
                               h_values[i]['upper'],
                               color='darkgreen', alpha=0.15)
                
                if show_mid_point and len(h_values[i]['mean']) > 100:
                    mid_point = len(h_values[i]['mean']) // 2
                    ax.axvline(x=mid_point, color='g', linestyle='--', alpha=0.7, label='midpoint')
                
                ax.set_ylabel("H Value", fontsize=12)
                ax.set_xlabel("Time", fontsize=12)
                ax.legend()
                ax.grid(alpha=0.3)
                print(f"Added H values plot for stim {i}")
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f"energy_kinematics_stim_{i}.png"), dpi=300, bbox_inches='tight')
        plt.close()
        print(f"Saved plot for stim {i}")
        
        # Save data to CSV files
        time_points = list(range(len(x_kin['mean'])))
        
        # X-coordinate CSV
        x_df = pd.DataFrame({
            'Time': time_points,
            'Mean_X_Position': x_kin['mean'],
            'Upper_CI': x_kin['upper'],
            'Lower_CI': x_kin['lower']
        })
        
        # Add other stim means for comparison
        for j, other_x_kin in enumerate(x_kinematics):
            if j != i and len(other_x_kin['mean']) == len(x_kin['mean']):
                x_df[f'Mean_X_Position_Stim_{j}'] = other_x_kin['mean']
        
        x_df.to_csv(os.path.join(output_dir, f"x_kinematics_stim_{i}.csv"), index=False)
        
        # Y-coordinate CSV
        y_df = pd.DataFrame({
            'Time': time_points,
            'Mean_Y_Position': y_kin['mean'],
            'Upper_CI': y_kin['upper'],
            'Lower_CI': y_kin['lower']
        })
        
        # Add other stim means for comparison
        for j, other_y_kin in enumerate(y_kinematics):
            if j != i and len(other_y_kin['mean']) == len(y_kin['mean']):
                y_df[f'Mean_Y_Position_Stim_{j}'] = other_y_kin['mean']
        
        y_df.to_csv(os.path.join(output_dir, f"y_kinematics_stim_{i}.csv"), index=False)
        
        # Z-coordinate CSV
        z_df = pd.DataFrame({
            'Time': time_points,
            'Mean_Z_Position': z_kin['mean'],
            'Upper_CI': z_kin['upper'],
            'Lower_CI': z_kin['lower']
        })
        
        # Add other stim means for comparison
        for j, other_z_kin in enumerate(z_kinematics):
            if j != i and len(other_z_kin['mean']) == len(z_kin['mean']):
                z_df[f'Mean_Z_Position_Stim_{j}'] = other_z_kin['mean']
        
        z_df.to_csv(os.path.join(output_dir, f"z_kinematics_stim_{i}.csv"), index=False)
        
        # Energy CSV
        eng_df = pd.DataFrame({
            'Time': time_points,
            'Mean_Energy': eng['mean'],
            'Upper_CI': eng['upper'],
            'Lower_CI': eng['lower'],
            'Critical_Energy': [critical_energy] * len(eng['mean']),
            'Mean_Energy_Overall': [np.mean(eng['mean'])] * len(eng['mean'])
        })
        
        # Add other stim means for comparison
        for j, other_eng in enumerate(energy):
            if j != i and len(other_eng['mean']) == len(eng['mean']):
                eng_df[f'Mean_Energy_Stim_{j}'] = other_eng['mean']
        
        eng_df.to_csv(os.path.join(output_dir, f"energy_stim_{i}.csv"), index=False)
        
        # Boltzmann Probability CSV
        boltzmann_df = pd.DataFrame({
            'Time': time_points,
            'Mean_Boltzmann_Prob': boltzmann_mean,
            'Upper_CI': boltzmann_upper,
            'Lower_CI': boltzmann_lower,
            'Temperature': [temperature] * len(boltzmann_mean)
        })
        
        # Add other stim Boltzmann probabilities for comparison
        for j, other_eng in enumerate(energy):
            if j != i and len(other_eng['mean']) == len(eng['mean']):
                other_boltzmann = np.exp(-np.array(other_eng['mean']) / temperature)
                boltzmann_df[f'Boltzmann_Prob_Stim_{j}'] = other_boltzmann
        
        boltzmann_df.to_csv(os.path.join(output_dir, f"boltzmann_probability_stim_{i}.csv"), index=False)
        
        # Firing rate CSV (if available)
        if firing_rate is not None:
            firing_data = firing_rate[i]
            firing_df = pd.DataFrame({
                'Time': time_points,
                'Mean_Firing_Rate': firing_data['mean'],
                'Upper_CI': firing_data['upper'],
                'Lower_CI': firing_data['lower']
            })
            firing_df.to_csv(os.path.join(output_dir, f"firing_rates_stim_{i}.csv"), index=False)
        
        # J values CSV (if applicable)
        if i == 1 and j_values is not None and len(j_values) > i:
            j_df = pd.DataFrame({
                'Time': time_points,
                'Mean_J': j_values[i]['mean'],
                'Upper_CI': j_values[i]['upper'],
                'Lower_CI': j_values[i]['lower']
            })
            j_df.to_csv(os.path.join(output_dir, f"j_values_stim_{i}.csv"), index=False)
        
        # H values CSV (if applicable)
        if i == 2 and h_values is not None and len(h_values) > i:
            h_df = pd.DataFrame({
                'Time': time_points,
                'Mean_H': h_values[i]['mean'],
                'Upper_CI': h_values[i]['upper'],
                'Lower_CI': h_values[i]['lower']
            })
            h_df.to_csv(os.path.join(output_dir, f"h_values_stim_{i}.csv"), index=False)
    
    print(f"Energy and kinematic (X, Y, Z) plots and CSVs saved to {output_dir}")

def plot_all_overlayed(stats, critical_energy, output_dir, title_prefix="", show_mid_point=True, temperature=1.0):
    """
    Create a single comprehensive plot with all stimuli overlayed.
    
    Parameters:
    -----------
    stats : dict
        Statistics dictionary containing x_kinematics, y_kinematics, z_kinematics, energy, firing_rate, j_values, h_values
    critical_energy : float
        Critical energy from phase transition analysis
    output_dir : str
        Directory to save plots
    title_prefix : str, optional
        Prefix for plot titles (default="")
    show_mid_point : bool, optional
        Whether to mark the middle point (default=True)
    temperature : float, optional
        Temperature parameter for Boltzmann probability calculation (default=1.0)
        
    Returns:
    --------
    None
    """
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import os
    
    os.makedirs(output_dir, exist_ok=True)
    
    x_kinematics = stats['x_kinematics']
    y_kinematics = stats['y_kinematics']
    z_kinematics = stats['z_kinematics']
    energy = stats['energy']
    firing_rate = stats.get('firing_rate', None)
    j_values = stats.get('j_values', None)
    h_values = stats.get('h_values', None)
    
    n_stim = len(x_kinematics)
    
    # Updated color scheme: red, blue, purple for first 3 stimuli, then additional colors
    colors = ['#FF0000', '#0000FF', '#9900FF', '#FF6B6B', '#4ECDC4', '#45B7D1']
    
    # Determine number of subplots needed
    n_plots = 6  # X, Y, Z, Velocity, Energy, Boltzmann
    if firing_rate is not None:
        n_plots += 1
    if j_values is not None:
        n_plots += 1
    if h_values is not None:
        n_plots += 1
    
    # Create figure with all subplots
    fig, axes = plt.subplots(n_plots, 1, figsize=(16, 4*n_plots), sharex=True)
    if n_plots == 1:
        axes = [axes]
    
    plot_idx = 0
    
    # X-coordinate overlay
    ax = axes[plot_idx]
    plot_idx += 1
    ax.set_title(f"{title_prefix} X-Coordinate Over Time - All Stimuli", fontsize=14, fontweight='bold')
    
    for i, x_kin in enumerate(x_kinematics):
        color = colors[i % len(colors)]
        ax.plot(x_kin['mean'], '-', linewidth=2.5, label=f'Stim {i}', color=color, alpha=0.9)
        ax.fill_between(range(len(x_kin['mean'])), x_kin['upper'], x_kin['lower'], 
                       color=color, alpha=0.15)
    
    if show_mid_point and len(x_kinematics[0]['mean']) > 100:
        mid_point = len(x_kinematics[0]['mean']) // 2
        ax.axvline(x=mid_point, color='gray', linestyle='--', alpha=0.5, linewidth=2, label='midpoint')
    
    ax.set_ylabel("X Position", fontsize=12, fontweight='bold')
    ax.legend(loc='best', fontsize=10)
    ax.grid(alpha=0.3)
    
    # Y-coordinate overlay
    ax = axes[plot_idx]
    plot_idx += 1
    ax.set_title(f"{title_prefix} Y-Coordinate Over Time - All Stimuli", fontsize=14, fontweight='bold')
    
    for i, y_kin in enumerate(y_kinematics):
        color = colors[i % len(colors)]
        ax.plot(y_kin['mean'], '-', linewidth=2.5, label=f'Stim {i}', color=color, alpha=0.9)
        ax.fill_between(range(len(y_kin['mean'])), y_kin['upper'], y_kin['lower'], 
                       color=color, alpha=0.15)
    
    if show_mid_point and len(y_kinematics[0]['mean']) > 100:
        mid_point = len(y_kinematics[0]['mean']) // 2
        ax.axvline(x=mid_point, color='gray', linestyle='--', alpha=0.5, linewidth=2, label='midpoint')
    
    ax.set_ylabel("Y Position", fontsize=12, fontweight='bold')
    ax.legend(loc='best', fontsize=10)
    ax.grid(alpha=0.3)
    
    # Z-coordinate overlay
    ax = axes[plot_idx]
    plot_idx += 1
    ax.set_title(f"{title_prefix} Z-Coordinate Over Time - All Stimuli", fontsize=14, fontweight='bold')
    
    for i, z_kin in enumerate(z_kinematics):
        color = colors[i % len(colors)]
        ax.plot(z_kin['mean'], '-', linewidth=2.5, label=f'Stim {i}', color=color, alpha=0.9)
        ax.fill_between(range(len(z_kin['mean'])), z_kin['upper'], z_kin['lower'], 
                       color=color, alpha=0.15)
    
    if show_mid_point and len(z_kinematics[0]['mean']) > 100:
        mid_point = len(z_kinematics[0]['mean']) // 2
        ax.axvline(x=mid_point, color='gray', linestyle='--', alpha=0.5, linewidth=2, label='midpoint')
    
    ax.set_ylabel("Z Position", fontsize=12, fontweight='bold')
    ax.legend(loc='best', fontsize=10)
    ax.grid(alpha=0.3)
    
    # Velocity overlay (computed from position data)
    ax = axes[plot_idx]
    plot_idx += 1
    ax.set_title(f"{title_prefix} Velocity (3D Magnitude) Over Time - All Stimuli", fontsize=14, fontweight='bold')
    
    velocity_data = []
    for i in range(n_stim):
        # Calculate velocity as the magnitude of change in position
        x_mean = np.array(x_kinematics[i]['mean'])
        y_mean = np.array(y_kinematics[i]['mean'])
        z_mean = np.array(z_kinematics[i]['mean'])
        
        # Compute differences (velocity = dx/dt, assuming dt=1)
        dx = np.diff(x_mean, prepend=x_mean[0])
        dy = np.diff(y_mean, prepend=y_mean[0])
        dz = np.diff(z_mean, prepend=z_mean[0])
        
        # Magnitude of velocity
        velocity = np.sqrt(dx**2 + dy**2 + dz**2)
        velocity_data.append(velocity)
        
        color = colors[i % len(colors)]
        ax.plot(velocity, '-', linewidth=2.5, label=f'Stim {i}', color=color, alpha=0.9)
    
    if show_mid_point and len(velocity_data[0]) > 100:
        mid_point = len(velocity_data[0]) // 2
        ax.axvline(x=mid_point, color='gray', linestyle='--', alpha=0.5, linewidth=2, label='midpoint')
    
    ax.set_ylabel("Velocity (3D)", fontsize=12, fontweight='bold')
    ax.legend(loc='best', fontsize=10)
    ax.grid(alpha=0.3)
    
    # Energy overlay with global extrema marked
    ax = axes[plot_idx]
    plot_idx += 1
    ax.set_title(f"{title_prefix} Energy Over Time - All Stimuli (Global Extrema Marked)", fontsize=14, fontweight='bold')
    
    for i, eng in enumerate(energy):
        color = colors[i % len(colors)]
        energy_mean = np.array(eng['mean'])
        ax.plot(energy_mean, '-', linewidth=2.5, label=f'Stim {i}', color=color, alpha=0.9)
        ax.fill_between(range(len(eng['mean'])), eng['upper'], eng['lower'], 
                       color=color, alpha=0.15)
        
        # Find global min and max only
        global_min_idx = np.argmin(energy_mean)
        global_max_idx = np.argmax(energy_mean)
        
        # Mark global minimum
        ax.plot(global_min_idx, energy_mean[global_min_idx], 'v', markersize=12, color=color, 
               markeredgecolor='black', markeredgewidth=2, alpha=0.9, 
               label=f'Stim {i} Min')
        
        # Mark global maximum
        ax.plot(global_max_idx, energy_mean[global_max_idx], '^', markersize=12, color=color, 
               markeredgecolor='black', markeredgewidth=2, alpha=0.9,
               label=f'Stim {i} Max')
    
    # # Mark critical energy
    # ax.axhline(y=critical_energy, color='red', linestyle='--', linewidth=2,
    #            label=f"Critical Energy = {critical_energy:.2f}", alpha=0.7)
    
    if show_mid_point and len(energy[0]['mean']) > 100:
        mid_point = len(energy[0]['mean']) // 2
        ax.axvline(x=mid_point, color='gray', linestyle='--', alpha=0.5, linewidth=2, label='midpoint')
    
    ax.set_ylabel("Energy", fontsize=12, fontweight='bold')
    ax.legend(loc='best', fontsize=9, ncol=2)
    ax.grid(alpha=0.3)
    
    # Boltzmann Probability overlay with global extrema marked
    ax = axes[plot_idx]
    plot_idx += 1
    ax.set_title(f"{title_prefix} Boltzmann Probability Over Time - All Stimuli (Global Extrema Marked)", fontsize=14, fontweight='bold')
    
    for i, eng in enumerate(energy):
        color = colors[i % len(colors)]
        boltzmann_mean = np.exp(-np.array(eng['mean']) / temperature)
        boltzmann_upper = np.exp(-np.array(eng['lower']) / temperature)
        boltzmann_lower = np.exp(-np.array(eng['upper']) / temperature)
        
        ax.plot(boltzmann_mean, '-', linewidth=2.5, label=f'Stim {i}', color=color, alpha=0.9)
        ax.fill_between(range(len(boltzmann_mean)), boltzmann_upper, boltzmann_lower, 
                       color=color, alpha=0.15)
        
        # Find global min and max only
        global_min_idx = np.argmin(boltzmann_mean)
        global_max_idx = np.argmax(boltzmann_mean)
        
        # Mark global minimum
        ax.plot(global_min_idx, boltzmann_mean[global_min_idx], 'v', markersize=12, color=color, 
               markeredgecolor='black', markeredgewidth=2, alpha=0.9,
               label=f'Stim {i} Min')
        
        # Mark global maximum
        ax.plot(global_max_idx, boltzmann_mean[global_max_idx], '^', markersize=12, color=color, 
               markeredgecolor='black', markeredgewidth=2, alpha=0.9,
               label=f'Stim {i} Max')
    
    if show_mid_point and len(energy[0]['mean']) > 100:
        mid_point = len(energy[0]['mean']) // 2
        ax.axvline(x=mid_point, color='gray', linestyle='--', alpha=0.5, linewidth=2, label='midpoint')
    
    ax.set_ylabel("Boltzmann Probability", fontsize=12, fontweight='bold')
    ax.legend(loc='best', fontsize=9, ncol=2)
    ax.grid(alpha=0.3)
    
    # Firing Rate overlay with global extrema marked (if available)
    if firing_rate is not None:
        ax = axes[plot_idx]
        plot_idx += 1
        ax.set_title(f"{title_prefix} Firing Rate Over Time - All Stimuli (Global Extrema Marked)", fontsize=14, fontweight='bold')
        
        for i, firing_data in enumerate(firing_rate):
            color = colors[i % len(colors)]
            firing_mean = np.array(firing_data['mean'])
            ax.plot(firing_mean, '-', linewidth=2.5, label=f'Stim {i}', color=color, alpha=0.9)
            ax.fill_between(range(len(firing_data['mean'])), 
                           firing_data['upper'],
                           firing_data['lower'],
                           color=color, alpha=0.15)
            
            # Find global min and max only
            global_min_idx = np.argmin(firing_mean)
            global_max_idx = np.argmax(firing_mean)
            
            # Mark global minimum
            ax.plot(global_min_idx, firing_mean[global_min_idx], 'v', markersize=12, color=color, 
                   markeredgecolor='black', markeredgewidth=2, alpha=0.9,
                   label=f'Stim {i} Min')
            
            # Mark global maximum
            ax.plot(global_max_idx, firing_mean[global_max_idx], '^', markersize=12, color=color, 
                   markeredgecolor='black', markeredgewidth=2, alpha=0.9,
                   label=f'Stim {i} Max')
        
        if show_mid_point and len(firing_rate[0]['mean']) > 100:
            mid_point = len(firing_rate[0]['mean']) // 2
            ax.axvline(x=mid_point, color='gray', linestyle='--', alpha=0.5, linewidth=2, label='midpoint')
        
        ax.set_ylabel("Firing Rate", fontsize=12, fontweight='bold')
        ax.legend(loc='best', fontsize=9, ncol=2)
        ax.grid(alpha=0.3)
    
    # J values overlay (if available)
    if j_values is not None:
        ax = axes[plot_idx]
        plot_idx += 1
        ax.set_title(f"{title_prefix} J Values (Local Interactions) Over Time - All Stimuli", fontsize=14, fontweight='bold')
        
        for i, j_val in enumerate(j_values):
            color = colors[i % len(colors)]
            ax.plot(j_val['mean'], '-', linewidth=2.5, label=f'Stim {i}', color=color, alpha=0.9)
            ax.fill_between(range(len(j_val['mean'])), 
                           j_val['upper'],
                           j_val['lower'],
                           color=color, alpha=0.15)
        
        if show_mid_point and len(j_values[0]['mean']) > 100:
            mid_point = len(j_values[0]['mean']) // 2
            ax.axvline(x=mid_point, color='gray', linestyle='--', alpha=0.5, linewidth=2, label='midpoint')
        
        ax.set_ylabel("J Value", fontsize=12, fontweight='bold')
        ax.legend(loc='best', fontsize=10)
        ax.grid(alpha=0.3)
    
    # H values overlay (if available)
    if h_values is not None:
        ax = axes[plot_idx]
        plot_idx += 1
        ax.set_title(f"{title_prefix} H Values (Local Fields) Over Time - All Stimuli", fontsize=14, fontweight='bold')
        
        for i, h_val in enumerate(h_values):
            color = colors[i % len(colors)]
            ax.plot(h_val['mean'], '-', linewidth=2.5, label=f'Stim {i}', color=color, alpha=0.9)
            ax.fill_between(range(len(h_val['mean'])), 
                           h_val['upper'],
                           h_val['lower'],
                           color=color, alpha=0.15)
        
        if show_mid_point and len(h_values[0]['mean']) > 100:
            mid_point = len(h_values[0]['mean']) // 2
            ax.axvline(x=mid_point, color='gray', linestyle='--', alpha=0.5, linewidth=2, label='midpoint')
        
        ax.set_ylabel("H Value", fontsize=12, fontweight='bold')
        ax.legend(loc='best', fontsize=10)
        ax.grid(alpha=0.3)
    
    # Set x-label on the last plot
    axes[-1].set_xlabel("Time", fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"all_stimuli_overlayed.png"), dpi=300, bbox_inches='tight')
    plt.close()
    
    # Save combined data to CSV including extrema information
    max_length = max(len(x_kinematics[i]['mean']) for i in range(n_stim))
    combined_df = pd.DataFrame({'Time': range(max_length)})
    
    # Store extrema information
    extrema_info = []
    
    # Add all metrics for all stimuli
    for i in range(n_stim):
        if len(x_kinematics[i]['mean']) == max_length:
            combined_df[f'Stim_{i}_X_Mean'] = x_kinematics[i]['mean']
            combined_df[f'Stim_{i}_X_Upper'] = x_kinematics[i]['upper']
            combined_df[f'Stim_{i}_X_Lower'] = x_kinematics[i]['lower']
            
            combined_df[f'Stim_{i}_Y_Mean'] = y_kinematics[i]['mean']
            combined_df[f'Stim_{i}_Y_Upper'] = y_kinematics[i]['upper']
            combined_df[f'Stim_{i}_Y_Lower'] = y_kinematics[i]['lower']
            
            combined_df[f'Stim_{i}_Z_Mean'] = z_kinematics[i]['mean']
            combined_df[f'Stim_{i}_Z_Upper'] = z_kinematics[i]['upper']
            combined_df[f'Stim_{i}_Z_Lower'] = z_kinematics[i]['lower']
            
            # Add velocity
            combined_df[f'Stim_{i}_Velocity'] = velocity_data[i]
            
            combined_df[f'Stim_{i}_Energy_Mean'] = energy[i]['mean']
            combined_df[f'Stim_{i}_Energy_Upper'] = energy[i]['upper']
            combined_df[f'Stim_{i}_Energy_Lower'] = energy[i]['lower']
            
            # Record energy extrema
            energy_mean = np.array(energy[i]['mean'])
            energy_min_idx = np.argmin(energy_mean)
            energy_max_idx = np.argmax(energy_mean)
            extrema_info.append({
                'Stimulus': i,
                'Metric': 'Energy',
                'Min_Index': energy_min_idx,
                'Min_Value': energy_mean[energy_min_idx],
                'Max_Index': energy_max_idx,
                'Max_Value': energy_mean[energy_max_idx]
            })
            
            boltzmann_mean = np.exp(-np.array(energy[i]['mean']) / temperature)
            boltzmann_upper = np.exp(-np.array(energy[i]['lower']) / temperature)
            boltzmann_lower = np.exp(-np.array(energy[i]['upper']) / temperature)
            combined_df[f'Stim_{i}_Boltzmann_Mean'] = boltzmann_mean
            combined_df[f'Stim_{i}_Boltzmann_Upper'] = boltzmann_upper
            combined_df[f'Stim_{i}_Boltzmann_Lower'] = boltzmann_lower
            
            # Record Boltzmann extrema
            boltz_min_idx = np.argmin(boltzmann_mean)
            boltz_max_idx = np.argmax(boltzmann_mean)
            extrema_info.append({
                'Stimulus': i,
                'Metric': 'Boltzmann',
                'Min_Index': boltz_min_idx,
                'Min_Value': boltzmann_mean[boltz_min_idx],
                'Max_Index': boltz_max_idx,
                'Max_Value': boltzmann_mean[boltz_max_idx]
            })
            
            if firing_rate is not None:
                combined_df[f'Stim_{i}_FiringRate_Mean'] = firing_rate[i]['mean']
                combined_df[f'Stim_{i}_FiringRate_Upper'] = firing_rate[i]['upper']
                combined_df[f'Stim_{i}_FiringRate_Lower'] = firing_rate[i]['lower']
                
                # Record firing rate extrema
                firing_mean = np.array(firing_rate[i]['mean'])
                firing_min_idx = np.argmin(firing_mean)
                firing_max_idx = np.argmax(firing_mean)
                extrema_info.append({
                    'Stimulus': i,
                    'Metric': 'FiringRate',
                    'Min_Index': firing_min_idx,
                    'Min_Value': firing_mean[firing_min_idx],
                    'Max_Index': firing_max_idx,
                    'Max_Value': firing_mean[firing_max_idx]
                })
            
            if j_values is not None:
                combined_df[f'Stim_{i}_J_Mean'] = j_values[i]['mean']
                combined_df[f'Stim_{i}_J_Upper'] = j_values[i]['upper']
                combined_df[f'Stim_{i}_J_Lower'] = j_values[i]['lower']
            
            if h_values is not None:
                combined_df[f'Stim_{i}_H_Mean'] = h_values[i]['mean']
                combined_df[f'Stim_{i}_H_Upper'] = h_values[i]['upper']
                combined_df[f'Stim_{i}_H_Lower'] = h_values[i]['lower']
    
    combined_df.to_csv(os.path.join(output_dir, f"all_stimuli_overlayed_data.csv"), index=False)
    
    # Save extrema information to separate CSV
    extrema_df = pd.DataFrame(extrema_info)
    extrema_df.to_csv(os.path.join(output_dir, f"extrema_summary.csv"), index=False)
    
    print(f"Overlayed plot for all stimuli saved to {output_dir}")
    print(f"Extrema summary saved to {output_dir}/extrema_summary.csv")

In [30]:
def calculate_statistics_from_dataframe(df, confidence=0.8, output_dir=None):
    """
    Calculate statistics across reaches for kinematic (x, y, z) and neural data from a dataframe.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with columns: reach_idx, stim, x, y, z, firing_rate, energy, j, h
        Each row represents a single time point for a specific reach in a specific stimulus
    confidence : float, optional
        Confidence level for intervals (default=0.8)
    output_dir : str, optional
        Directory to save statistics results
        
    Returns:
    --------
    dict
        Dictionary containing mean and confidence intervals for each measure
    """
    from energy_over_time_script.utils import mean_confidence_interval
    
    # Get unique stimuli
    stimuli = sorted(df['stim'].unique())
    
    # Initialize containers for results
    x_kinematics_stats = []
    y_kinematics_stats = []
    z_kinematics_stats = []
    energy_stats = []
    firing_rate_stats = []
    j_values_stats = []
    h_values_stats = []
    
    # Calculate statistics for each stimulation condition
    for stim in stimuli:
        # Filter data for this stimulus
        stim_data = df[df['stim'] == stim].copy()
        
        # Group by time index (assuming rows are ordered by time)
        # We need to identify unique time points across all reaches
        stim_data['time_idx'] = stim_data.groupby('reach_idx').cumcount()
        
        # Get the maximum time length
        max_time = stim_data['time_idx'].max() + 1
        
        # Initialize lists for this stimulus
        x_mean, x_lower, x_upper = [], [], []
        y_mean, y_lower, y_upper = [], [], []
        z_mean, z_lower, z_upper = [], [], []
        energy_mean, energy_lower, energy_upper = [], [], []
        firing_mean, firing_lower, firing_upper = [], [], []
        j_mean, j_lower, j_upper = [], [], []
        h_mean, h_lower, h_upper = [], [], []
        
        # Calculate statistics for each time point
        for t in range(max_time):
            time_data = stim_data[stim_data['time_idx'] == t]
            
            if len(time_data) > 0:
                # X-coordinate statistics
                x_m, x_ml, x_mu = mean_confidence_interval(time_data['x'].values, confidence)
                x_mean.append(x_m)
                x_lower.append(x_ml)
                x_upper.append(x_mu)
                
                # Y-coordinate statistics
                y_m, y_ml, y_mu = mean_confidence_interval(time_data['y'].values, confidence)
                y_mean.append(y_m)
                y_lower.append(y_ml)
                y_upper.append(y_mu)
                
                # Z-coordinate statistics
                z_m, z_ml, z_mu = mean_confidence_interval(time_data['z'].values, confidence)
                z_mean.append(z_m)
                z_lower.append(z_ml)
                z_upper.append(z_mu)
                
                # Energy statistics
                e_m, e_ml, e_mu = mean_confidence_interval(time_data['energy'].values, confidence)
                energy_mean.append(e_m)
                energy_lower.append(e_ml)
                energy_upper.append(e_mu)
                
                # Firing rate statistics
                f_m, f_ml, f_mu = mean_confidence_interval(time_data['firing_rate'].values, confidence)
                firing_mean.append(f_m)
                firing_lower.append(f_ml)
                firing_upper.append(f_mu)
                
                # J statistics
                j_m, j_ml, j_mu = mean_confidence_interval(time_data['j'].values, confidence)
                j_mean.append(j_m)
                j_lower.append(j_ml)
                j_upper.append(j_mu)
                
                # H statistics
                h_m, h_ml, h_mu = mean_confidence_interval(time_data['h'].values, confidence)
                h_mean.append(h_m)
                h_lower.append(h_ml)
                h_upper.append(h_mu)
        
        # Store statistics for this stimulus
        x_kinematics_stats.append({
            'mean': x_mean,
            'lower': x_lower,
            'upper': x_upper
        })
        
        y_kinematics_stats.append({
            'mean': y_mean,
            'lower': y_lower,
            'upper': y_upper
        })
        
        z_kinematics_stats.append({
            'mean': z_mean,
            'lower': z_lower,
            'upper': z_upper
        })
        
        energy_stats.append({
            'mean': energy_mean,
            'lower': energy_lower,
            'upper': energy_upper
        })
        
        firing_rate_stats.append({
            'mean': firing_mean,
            'lower': firing_lower,
            'upper': firing_upper
        })
        
        j_values_stats.append({
            'mean': j_mean,
            'lower': j_lower,
            'upper': j_upper
        })
        
        h_values_stats.append({
            'mean': h_mean,
            'lower': h_lower,
            'upper': h_upper
        })
        
        # Save statistics to CSV and create plots if output directory is provided
        if output_dir:
            os.makedirs(output_dir, exist_ok=True)
            time_points = list(range(len(x_mean)))
            
            # Save X-coordinate statistics
            x_stats_df = pd.DataFrame({
                'Time': time_points,
                'Mean': x_mean,
                'Lower_CI': x_lower,
                'Upper_CI': x_upper
            })
            x_stats_df.to_csv(os.path.join(output_dir, f"x_kinematics_stats_stim_{stim}.csv"), index=False)
            
            # Save Y-coordinate statistics
            y_stats_df = pd.DataFrame({
                'Time': time_points,
                'Mean': y_mean,
                'Lower_CI': y_lower,
                'Upper_CI': y_upper
            })
            y_stats_df.to_csv(os.path.join(output_dir, f"y_kinematics_stats_stim_{stim}.csv"), index=False)
            
            # Save Z-coordinate statistics
            z_stats_df = pd.DataFrame({
                'Time': time_points,
                'Mean': z_mean,
                'Lower_CI': z_lower,
                'Upper_CI': z_upper
            })
            z_stats_df.to_csv(os.path.join(output_dir, f"z_kinematics_stats_stim_{stim}.csv"), index=False)
            
            # Save Energy statistics
            energy_stats_df = pd.DataFrame({
                'Time': time_points,
                'Mean': energy_mean,
                'Lower_CI': energy_lower,
                'Upper_CI': energy_upper
            })
            energy_stats_df.to_csv(os.path.join(output_dir, f"energy_stats_stim_{stim}.csv"), index=False)
            
            # Save Firing Rate statistics
            firing_stats_df = pd.DataFrame({
                'Time': time_points,
                'Mean': firing_mean,
                'Lower_CI': firing_lower,
                'Upper_CI': firing_upper
            })
            firing_stats_df.to_csv(os.path.join(output_dir, f"firing_rate_stats_stim_{stim}.csv"), index=False)
            
            # Save J statistics
            j_stats_df = pd.DataFrame({
                'Time': time_points,
                'Mean': j_mean,
                'Lower_CI': j_lower,
                'Upper_CI': j_upper
            })
            j_stats_df.to_csv(os.path.join(output_dir, f"j_stats_stim_{stim}.csv"), index=False)
            
            # Save H statistics
            h_stats_df = pd.DataFrame({
                'Time': time_points,
                'Mean': h_mean,
                'Lower_CI': h_lower,
                'Upper_CI': h_upper
            })
            h_stats_df.to_csv(os.path.join(output_dir, f"h_stats_stim_{stim}.csv"), index=False)
            
            # Create stacked plots for X, Y, Z coordinates, Energy, and Firing Rate
            fig, axs = plt.subplots(5, 1, figsize=(12, 20), sharex=True)
            
            # X-coordinate plot
            axs[0].plot(time_points, x_mean, 'b-', linewidth=2, label='Mean X')
            axs[0].fill_between(time_points, x_lower, x_upper, alpha=0.3, color='blue', label=f'{int(confidence*100)}% CI')
            axs[0].set_ylabel('X Coordinate')
            axs[0].set_title(f'X-Coordinate Statistics - Stimulus {stim}')
            axs[0].legend()
            axs[0].grid(True, alpha=0.3)
            
            # Y-coordinate plot
            axs[1].plot(time_points, y_mean, 'g-', linewidth=2, label='Mean Y')
            axs[1].fill_between(time_points, y_lower, y_upper, alpha=0.3, color='green', label=f'{int(confidence*100)}% CI')
            axs[1].set_ylabel('Y Coordinate')
            axs[1].set_title(f'Y-Coordinate Statistics - Stimulus {stim}')
            axs[1].legend()
            axs[1].grid(True, alpha=0.3)
            
            # Z-coordinate plot
            axs[2].plot(time_points, z_mean, 'r-', linewidth=2, label='Mean Z')
            axs[2].fill_between(time_points, z_lower, z_upper, alpha=0.3, color='red', label=f'{int(confidence*100)}% CI')
            axs[2].set_ylabel('Z Coordinate')
            axs[2].set_title(f'Z-Coordinate Statistics - Stimulus {stim}')
            axs[2].legend()
            axs[2].grid(True, alpha=0.3)
            
            # Energy plot
            axs[3].plot(time_points, energy_mean, 'purple', linewidth=2, label='Mean Energy')
            axs[3].fill_between(time_points, energy_lower, energy_upper, alpha=0.3, color='purple', label=f'{int(confidence*100)}% CI')
            axs[3].set_ylabel('Energy')
            axs[3].set_title(f'Energy Statistics - Stimulus {stim}')
            axs[3].legend()
            axs[3].grid(True, alpha=0.3)
            
            # Firing Rate plot
            axs[4].plot(time_points, firing_mean, 'orange', linewidth=2, label='Mean Firing Rate')
            axs[4].fill_between(time_points, firing_lower, firing_upper, alpha=0.3, color='orange', label=f'{int(confidence*100)}% CI')
            axs[4].set_ylabel('Firing Rate')
            axs[4].set_xlabel('Time Points')
            axs[4].set_title(f'Firing Rate Statistics - Stimulus {stim}')
            axs[4].legend()
            axs[4].grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, f"stacked_statistics_stim_{stim}.png"), dpi=300, bbox_inches='tight')
            plt.close()
    
    return {
        'x_kinematics': x_kinematics_stats,
        'y_kinematics': y_kinematics_stats,
        'z_kinematics': z_kinematics_stats,
        'energy': energy_stats,
        'firing_rate': firing_rate_stats,
        'j_values': j_values_stats,
        'h_values': h_values_stats
    }


def plot_energy_across_time_from_stats(stats, critical_energy, output_dir, title_prefix="", show_mid_point=True, window_size=10, temperature=1.0):
    """
    Plot energy and kinematic data (x, y, z coordinates) across time from pre-computed statistics.
    
    Parameters:
    -----------
    stats : dict
        Statistics dictionary containing x_kinematics, y_kinematics, z_kinematics, energy, firing_rate, j_values, h_values
    critical_energy : float
        Critical energy from phase transition analysis
    output_dir : str
        Directory to save plots
    title_prefix : str, optional
        Prefix for plot titles (default="")
    show_mid_point : bool, optional
        Whether to mark the middle point (default=True)
    window_size : int, optional
        Size of the sliding window for firing rate calculation (default=10)
    temperature : float, optional
        Temperature parameter for Boltzmann probability calculation (default=1.0)
        
    Returns:
    --------
    None
    """
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import os
    
    os.makedirs(output_dir, exist_ok=True)
    
    x_kinematics = stats['x_kinematics']
    y_kinematics = stats['y_kinematics']
    z_kinematics = stats['z_kinematics']
    energy = stats['energy']
    firing_rate = stats.get('firing_rate', None)
    j_values = stats.get('j_values', None)
    h_values = stats.get('h_values', None)
    
    # Debug: print what we have
    print(f"j_values is None: {j_values is None}")
    print(f"h_values is None: {h_values is None}")
    if j_values is not None:
        print(f"j_values length: {len(j_values)}")
    if h_values is not None:
        print(f"h_values length: {len(h_values)}")
    
    # Create overlapped firing rate plot (all stimuli together)
    if firing_rate is not None:
        plt.figure(figsize=(14, 6))
        
        # Updated color scheme: red, blue, purple for first 3 stimuli
        colors = ['#FF0000', '#0000FF', '#9900FF', '#FF6B6B', '#4ECDC4', '#45B7D1']
        
        for i, firing_data in enumerate(firing_rate):
            color = colors[i % len(colors)]
            plt.plot(firing_data['mean'], '-', linewidth=2, 
                    label=f'Stim_{i} Firing Rate', color=color, alpha=0.8)
            plt.fill_between(range(len(firing_data['mean'])), 
                           firing_data['lower'],
                           firing_data['upper'],
                           color=color, alpha=0.15)
        
        # Mark midpoint if requested
        if show_mid_point and len(firing_rate[0]['mean']) > 100:
            mid_point = len(firing_rate[0]['mean']) // 2
            plt.axvline(x=mid_point, color='g', linestyle='--', alpha=0.7, label='midpoint')
        
        plt.xlabel("Time")
        plt.ylabel("Firing Rate")
        plt.title(f"{title_prefix} Firing Rate Comparison Across All Stimuli")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f"firing_rate_all_stimuli.png"), dpi=300, bbox_inches='tight')
        plt.close()
        
        # Save combined firing rate data to CSV
        combined_firing_df = pd.DataFrame({'Time': range(len(firing_rate[0]['mean']))})
        for i, firing_data in enumerate(firing_rate):
            combined_firing_df[f'Stim_{i}_Mean'] = firing_data['mean']
            combined_firing_df[f'Stim_{i}_Upper_CI'] = firing_data['upper']
            combined_firing_df[f'Stim_{i}_Lower_CI'] = firing_data['lower']
        combined_firing_df.to_csv(os.path.join(output_dir, f"firing_rate_all_stimuli.csv"), index=False)
    
    # Plot individual stimulus data
    for i, (x_kin, y_kin, z_kin, eng) in enumerate(zip(x_kinematics, y_kinematics, z_kinematics, energy)):
        # Determine layout: we'll use 2 columns for stim 1 and 2
        # Left column: X, Y, Z, Energy, Boltzmann
        # Right column: J values (stim 1), H values (stim 2), empty otherwise
        
        n_rows = 5  # X, Y, Z, Energy, Boltzmann
        
        # Check if we need the right column for this stimulus
        has_right_column = False
        if i == 1 and j_values is not None and len(j_values) > i:
            has_right_column = True
            print(f"Stim {i}: Will show J values in right column")
        elif i == 2 and h_values is not None and len(h_values) > i:
            has_right_column = True
            print(f"Stim {i}: Will show H values in right column")
        else:
            print(f"Stim {i}: No right column")
        
        if has_right_column:
            fig = plt.figure(figsize=(20, 3*n_rows))
            gs = fig.add_gridspec(n_rows, 2, hspace=0.3, wspace=0.3)
        else:
            fig = plt.figure(figsize=(12, 3*n_rows))
            gs = fig.add_gridspec(n_rows, 1, hspace=0.3)
        
        # X-coordinate subplot (left column, row 0)
        ax = fig.add_subplot(gs[0, 0])
        ax.set_title(f"{title_prefix} X-Coordinate Over Time, Stim_{i}")
        
        ax.plot(x_kin['mean'], '-r', linewidth=2, label='mean X')
        ax.plot(x_kin['upper'], '-b', label='upper CI', alpha=0.5)
        ax.plot(x_kin['lower'], '-b', label='lower CI', alpha=0.5)
        
        # Plot other stimulus means for comparison if available
        for j, other_x_kin in enumerate(x_kinematics):
            if j != i:
                ax.plot(other_x_kin['mean'], '-', label=f"mean X stim_{j}", alpha=0.7)
        
        # Fill between confidence intervals
        ax.fill_between(list(range(len(x_kin['mean']))), x_kin['upper'], x_kin['lower'], 
                         color="blue", alpha=0.15)
        
        # Mark midpoint if requested
        if show_mid_point and len(x_kin['mean']) > 100:
            mid_point = len(x_kin['mean']) // 2
            ax.axvline(x=mid_point, color='g', linestyle='--', alpha=0.7, label='midpoint')
        
        ax.set_ylabel("X Position")
        ax.legend()
        ax.grid(alpha=0.3)
        
        # Y-coordinate subplot (left column, row 1)
        ax = fig.add_subplot(gs[1, 0])
        ax.set_title(f"{title_prefix} Y-Coordinate Over Time, Stim_{i}")
        
        ax.plot(y_kin['mean'], '-g', linewidth=2, label='mean Y')
        ax.plot(y_kin['upper'], '-b', label='upper CI', alpha=0.5)
        ax.plot(y_kin['lower'], '-b', label='lower CI', alpha=0.5)
        
        # Plot other stimulus means for comparison if available
        for j, other_y_kin in enumerate(y_kinematics):
            if j != i:
                ax.plot(other_y_kin['mean'], '-', label=f"mean Y stim_{j}", alpha=0.7)
        
        # Fill between confidence intervals
        ax.fill_between(list(range(len(y_kin['mean']))), y_kin['upper'], y_kin['lower'], 
                         color="green", alpha=0.15)
        
        # Mark midpoint if requested
        if show_mid_point and len(y_kin['mean']) > 100:
            mid_point = len(y_kin['mean']) // 2
            ax.axvline(x=mid_point, color='g', linestyle='--', alpha=0.7, label='midpoint')
        
        ax.set_ylabel("Y Position")
        ax.legend()
        ax.grid(alpha=0.3)
        
        # Z-coordinate subplot (left column, row 2)
        ax = fig.add_subplot(gs[2, 0])
        ax.set_title(f"{title_prefix} Z-Coordinate Over Time, Stim_{i}")
        
        ax.plot(z_kin['mean'], '-r', linewidth=2, label='mean Z', color='red')
        ax.plot(z_kin['upper'], '-b', label='upper CI', alpha=0.5)
        ax.plot(z_kin['lower'], '-b', label='lower CI', alpha=0.5)
        
        # Plot other stimulus means for comparison if available
        for j, other_z_kin in enumerate(z_kinematics):
            if j != i:
                ax.plot(other_z_kin['mean'], '-', label=f"mean Z stim_{j}", alpha=0.7)
        
        # Fill between confidence intervals
        ax.fill_between(list(range(len(z_kin['mean']))), z_kin['upper'], z_kin['lower'], 
                         color="red", alpha=0.15)
        
        # Mark midpoint if requested
        if show_mid_point and len(z_kin['mean']) > 100:
            mid_point = len(z_kin['mean']) // 2
            ax.axvline(x=mid_point, color='g', linestyle='--', alpha=0.7, label='midpoint')
        
        ax.set_ylabel("Z Position")
        ax.legend()
        ax.grid(alpha=0.3)
        
        # Energy subplot (left column, row 3)
        ax = fig.add_subplot(gs[3, 0])
        ax.set_title(f"{title_prefix} Energy of Neural Activity Over Time, Stim_{i}")
        
        # Fill between confidence intervals
        ax.fill_between(list(range(len(eng['mean']))), eng['upper'], eng['lower'], 
                         color="purple", alpha=0.15)
        
        ax.plot(eng['mean'], '-', linewidth=2, label='mean Energy', color='purple')
        
        # Plot other stimulus means for comparison if available
        for j, other_eng in enumerate(energy):
            if j != i:
                ax.plot(other_eng['mean'], '-', label=f"mean Energy stim_{j}", alpha=0.7)
        
        # Mark critical energy and mean energy
        ax.axhline(y=critical_energy, color='r', linestyle='--',
                   label=f"Critical Energy = {critical_energy:.2f}")
        ax.axhline(y=np.mean(eng['mean']), color='b', linestyle='-.',
                   label=f"Mean Energy = {np.mean(eng['mean']):.2f}")
        
        # Mark midpoint if requested
        if show_mid_point and len(eng['mean']) > 100:
            mid_point = len(eng['mean']) // 2
            ax.axvline(x=mid_point, color='g', linestyle='--', alpha=0.7, label='midpoint')
        
        ax.set_ylabel("Energy")
        ax.legend()
        ax.grid(alpha=0.3)
        
        # Boltzmann Probability subplot (left column, row 4)
        ax = fig.add_subplot(gs[4, 0])
        ax.set_title(f"{title_prefix} Boltzmann Probability Over Time, Stim_{i}")
        
        # Calculate Boltzmann probabilities: exp(-Energy/T)
        boltzmann_mean = np.exp(-np.array(eng['mean']) / temperature)
        boltzmann_upper = np.exp(-np.array(eng['lower']) / temperature)  # Note: lower energy -> higher probability
        boltzmann_lower = np.exp(-np.array(eng['upper']) / temperature)  # Note: higher energy -> lower probability
        
        # Fill between confidence intervals
        ax.fill_between(list(range(len(boltzmann_mean))), boltzmann_upper, boltzmann_lower, 
                         color="teal", alpha=0.15)
        
        ax.plot(boltzmann_mean, '-', linewidth=2, label='mean Boltzmann Prob', color='teal')
        
        # Plot other stimulus Boltzmann probabilities for comparison if available
        for j, other_eng in enumerate(energy):
            if j != i:
                other_boltzmann = np.exp(-np.array(other_eng['mean']) / temperature)
                ax.plot(other_boltzmann, '-', label=f"Boltzmann Prob stim_{j}", alpha=0.7)
        
        # Mark midpoint if requested
        if show_mid_point and len(boltzmann_mean) > 100:
            mid_point = len(boltzmann_mean) // 2
            ax.axvline(x=mid_point, color='g', linestyle='--', alpha=0.7, label='midpoint')
        
        ax.set_xlabel("Time")
        ax.set_ylabel("Boltzmann Probability")
        ax.legend()
        ax.grid(alpha=0.3)
        
        # Right column plots
        if has_right_column:
            if i == 1 and j_values is not None and len(j_values) > i:
                # J values plot for stimulus 1 (right column, spanning rows 0-1)
                ax = fig.add_subplot(gs[0:2, 1])
                ax.set_title(f"{title_prefix} J Values (Local Interactions), Stim_{i}")
                
                ax.plot(j_values[i]['mean'], '-', linewidth=3, label=f"mean J", color='darkblue')
                ax.fill_between(range(len(j_values[i]['mean'])), 
                               j_values[i]['lower'],
                               j_values[i]['upper'],
                               color='darkblue', alpha=0.15)
                
                if show_mid_point and len(j_values[i]['mean']) > 100:
                    mid_point = len(j_values[i]['mean']) // 2
                    ax.axvline(x=mid_point, color='g', linestyle='--', alpha=0.7, label='midpoint')
                
                ax.set_ylabel("J Value", fontsize=12)
                ax.set_xlabel("Time", fontsize=12)
                ax.legend()
                ax.grid(alpha=0.3)
                print(f"Added J values plot for stim {i}")
                
            elif i == 2 and h_values is not None and len(h_values) > i:
                # H values plot for stimulus 2 (right column, spanning rows 0-1)
                ax = fig.add_subplot(gs[0:2, 1])
                ax.set_title(f"{title_prefix} H Values (Local Fields), Stim_{i}")
                
                ax.plot(h_values[i]['mean'], '-', linewidth=3, label=f"mean H", color='darkgreen')
                ax.fill_between(range(len(h_values[i]['mean'])), 
                               h_values[i]['lower'],
                               h_values[i]['upper'],
                               color='darkgreen', alpha=0.15)
                
                if show_mid_point and len(h_values[i]['mean']) > 100:
                    mid_point = len(h_values[i]['mean']) // 2
                    ax.axvline(x=mid_point, color='g', linestyle='--', alpha=0.7, label='midpoint')
                
                ax.set_ylabel("H Value", fontsize=12)
                ax.set_xlabel("Time", fontsize=12)
                ax.legend()
                ax.grid(alpha=0.3)
                print(f"Added H values plot for stim {i}")
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f"energy_kinematics_stim_{i}.png"), dpi=300, bbox_inches='tight')
        plt.close()
        print(f"Saved plot for stim {i}")
        
        # Save data to CSV files
        time_points = list(range(len(x_kin['mean'])))
        
        # X-coordinate CSV
        x_df = pd.DataFrame({
            'Time': time_points,
            'Mean_X_Position': x_kin['mean'],
            'Upper_CI': x_kin['upper'],
            'Lower_CI': x_kin['lower']
        })
        
        # Add other stim means for comparison
        for j, other_x_kin in enumerate(x_kinematics):
            if j != i and len(other_x_kin['mean']) == len(x_kin['mean']):
                x_df[f'Mean_X_Position_Stim_{j}'] = other_x_kin['mean']
        
        x_df.to_csv(os.path.join(output_dir, f"x_kinematics_stim_{i}.csv"), index=False)
        
        # Y-coordinate CSV
        y_df = pd.DataFrame({
            'Time': time_points,
            'Mean_Y_Position': y_kin['mean'],
            'Upper_CI': y_kin['upper'],
            'Lower_CI': y_kin['lower']
        })
        
        # Add other stim means for comparison
        for j, other_y_kin in enumerate(y_kinematics):
            if j != i and len(other_y_kin['mean']) == len(y_kin['mean']):
                y_df[f'Mean_Y_Position_Stim_{j}'] = other_y_kin['mean']
        
        y_df.to_csv(os.path.join(output_dir, f"y_kinematics_stim_{i}.csv"), index=False)
        
        # Z-coordinate CSV
        z_df = pd.DataFrame({
            'Time': time_points,
            'Mean_Z_Position': z_kin['mean'],
            'Upper_CI': z_kin['upper'],
            'Lower_CI': z_kin['lower']
        })
        
        # Add other stim means for comparison
        for j, other_z_kin in enumerate(z_kinematics):
            if j != i and len(other_z_kin['mean']) == len(z_kin['mean']):
                z_df[f'Mean_Z_Position_Stim_{j}'] = other_z_kin['mean']
        
        z_df.to_csv(os.path.join(output_dir, f"z_kinematics_stim_{i}.csv"), index=False)
        
        # Energy CSV
        eng_df = pd.DataFrame({
            'Time': time_points,
            'Mean_Energy': eng['mean'],
            'Upper_CI': eng['upper'],
            'Lower_CI': eng['lower'],
            'Critical_Energy': [critical_energy] * len(eng['mean']),
            'Mean_Energy_Overall': [np.mean(eng['mean'])] * len(eng['mean'])
        })
        
        # Add other stim means for comparison
        for j, other_eng in enumerate(energy):
            if j != i and len(other_eng['mean']) == len(eng['mean']):
                eng_df[f'Mean_Energy_Stim_{j}'] = other_eng['mean']
        
        eng_df.to_csv(os.path.join(output_dir, f"energy_stim_{i}.csv"), index=False)
        
        # Boltzmann Probability CSV
        boltzmann_df = pd.DataFrame({
            'Time': time_points,
            'Mean_Boltzmann_Prob': boltzmann_mean,
            'Upper_CI': boltzmann_upper,
            'Lower_CI': boltzmann_lower,
            'Temperature': [temperature] * len(boltzmann_mean)
        })
        
        # Add other stim Boltzmann probabilities for comparison
        for j, other_eng in enumerate(energy):
            if j != i and len(other_eng['mean']) == len(eng['mean']):
                other_boltzmann = np.exp(-np.array(other_eng['mean']) / temperature)
                boltzmann_df[f'Boltzmann_Prob_Stim_{j}'] = other_boltzmann
        
        boltzmann_df.to_csv(os.path.join(output_dir, f"boltzmann_probability_stim_{i}.csv"), index=False)
        
        # Firing rate CSV (if available)
        if firing_rate is not None:
            firing_data = firing_rate[i]
            firing_df = pd.DataFrame({
                'Time': time_points,
                'Mean_Firing_Rate': firing_data['mean'],
                'Upper_CI': firing_data['upper'],
                'Lower_CI': firing_data['lower']
            })
            firing_df.to_csv(os.path.join(output_dir, f"firing_rates_stim_{i}.csv"), index=False)
        
        # J values CSV (if applicable)
        if i == 1 and j_values is not None and len(j_values) > i:
            j_df = pd.DataFrame({
                'Time': time_points,
                'Mean_J': j_values[i]['mean'],
                'Upper_CI': j_values[i]['upper'],
                'Lower_CI': j_values[i]['lower']
            })
            j_df.to_csv(os.path.join(output_dir, f"j_values_stim_{i}.csv"), index=False)
        
        # H values CSV (if applicable)
        if i == 2 and h_values is not None and len(h_values) > i:
            h_df = pd.DataFrame({
                'Time': time_points,
                'Mean_H': h_values[i]['mean'],
                'Upper_CI': h_values[i]['upper'],
                'Lower_CI': h_values[i]['lower']
            })
            h_df.to_csv(os.path.join(output_dir, f"h_values_stim_{i}.csv"), index=False)
    
    print(f"Energy and kinematic (X, Y, Z) plots and CSVs saved to {output_dir}")


def plot_all_overlayed(stats, critical_energy, output_dir, title_prefix="", show_mid_point=True, temperature=1.0, 
                       extrema_range_start=350, extrema_range_end=450):
    """
    Create a single comprehensive plot with all stimuli overlayed.
    
    Parameters:
    -----------
    stats : dict
        Statistics dictionary containing x_kinematics, y_kinematics, z_kinematics, energy, firing_rate, j_values, h_values
    critical_energy : float
        Critical energy from phase transition analysis
    output_dir : str
        Directory to save plots
    title_prefix : str, optional
        Prefix for plot titles (default="")
    show_mid_point : bool, optional
        Whether to mark the middle point (default=True)
    temperature : float, optional
        Temperature parameter for Boltzmann probability calculation (default=1.0)
    extrema_range_start : int, optional
        Start index for finding extrema (default=350)
    extrema_range_end : int, optional
        End index for finding extrema (default=450)
        
    Returns:
    --------
    None
    """
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import os
    
    os.makedirs(output_dir, exist_ok=True)
    
    x_kinematics = stats['x_kinematics']
    y_kinematics = stats['y_kinematics']
    z_kinematics = stats['z_kinematics']
    energy = stats['energy']
    firing_rate = stats.get('firing_rate', None)
    j_values = stats.get('j_values', None)
    h_values = stats.get('h_values', None)
    
    n_stim = len(x_kinematics)
    
    # Updated color scheme: red, blue, purple for first 3 stimuli, then additional colors
    colors = ['#FF0000', '#0000FF', '#9900FF', '#FF6B6B', '#4ECDC4', '#45B7D1']
    
    # Determine number of subplots needed
    n_plots = 6  # X, Y, Z, Velocity, Energy, Boltzmann
    if firing_rate is not None:
        n_plots += 1
    if j_values is not None:
        n_plots += 1
    if h_values is not None:
        n_plots += 1
    
    # Create figure with all subplots
    fig, axes = plt.subplots(n_plots, 1, figsize=(16, 4*n_plots), sharex=True)
    if n_plots == 1:
        axes = [axes]
    
    plot_idx = 0
    
    # Helper function to find extrema within a constrained range
    def find_extrema_in_range(data, start_idx, end_idx):
        """Find min and max indices within the specified range."""
        # Ensure indices are within bounds
        start_idx = max(0, start_idx)
        end_idx = min(len(data), end_idx)
        
        if start_idx >= end_idx or start_idx >= len(data):
            # Fallback to global if range is invalid
            return np.argmin(data), np.argmax(data)
        
        # Get the slice and find extrema
        data_slice = data[start_idx:end_idx]
        local_min_idx = np.argmin(data_slice)
        local_max_idx = np.argmax(data_slice)
        
        # Convert back to global indices
        global_min_idx = start_idx + local_min_idx
        global_max_idx = start_idx + local_max_idx
        
        return global_min_idx, global_max_idx
    
    print(f"Finding extrema within range [{extrema_range_start}, {extrema_range_end}]")
    
    # Pre-compute velocity data and find max velocity index within range for each stimulus
    velocity_data = []
    max_velocity_indices = []
    for i in range(n_stim):
        # Calculate velocity as the magnitude of change in position
        x_mean = np.array(x_kinematics[i]['mean'])
        y_mean = np.array(y_kinematics[i]['mean'])
        z_mean = np.array(z_kinematics[i]['mean'])
        
        # Compute differences (velocity = dx/dt, assuming dt=1)
        dx = np.diff(x_mean, prepend=x_mean[0])
        dy = np.diff(y_mean, prepend=y_mean[0])
        dz = np.diff(z_mean, prepend=z_mean[0])
        
        # Magnitude of velocity
        velocity = np.sqrt(dx**2 + dy**2 + dz**2)
        velocity_data.append(velocity)
        
        # Find max velocity index within the specified range
        _, max_vel_idx = find_extrema_in_range(velocity, extrema_range_start, extrema_range_end)
        max_velocity_indices.append(max_vel_idx)
        print(f"Stim {i}: Max velocity index = {max_vel_idx} (value = {velocity[max_vel_idx]:.4f})")
    
    # X-coordinate overlay
    ax = axes[plot_idx]
    plot_idx += 1
    ax.set_title(f"{title_prefix} X-Coordinate Over Time - All Stimuli", fontsize=14, fontweight='bold')
    
    for i, x_kin in enumerate(x_kinematics):
        color = colors[i % len(colors)]
        ax.plot(x_kin['mean'], '-', linewidth=2.5, label=f'Stim {i}', color=color, alpha=0.9)
        ax.fill_between(range(len(x_kin['mean'])), x_kin['upper'], x_kin['lower'], 
                       color=color, alpha=0.15)
    
    if show_mid_point and len(x_kinematics[0]['mean']) > 100:
        mid_point = len(x_kinematics[0]['mean']) // 2
        ax.axvline(x=mid_point, color='gray', linestyle='--', alpha=0.5, linewidth=2, label='midpoint')
    
    # Add shaded region for extrema search range
    ax.axvspan(extrema_range_start, extrema_range_end, alpha=0.1, color='yellow', label='Extrema Search Range')
    
    # Add vertical lines for max velocity indices
    for i, max_vel_idx in enumerate(max_velocity_indices):
        color = colors[i % len(colors)]
        ax.axvline(x=max_vel_idx, color=color, linestyle=':', linewidth=2.5, alpha=0.8, 
                   label=f'Stim {i} Max Vel (idx={max_vel_idx})')
    
    ax.set_ylabel("X Position", fontsize=12, fontweight='bold')
    ax.legend(loc='best', fontsize=10)
    ax.grid(alpha=0.3)
    
    # Y-coordinate overlay
    ax = axes[plot_idx]
    plot_idx += 1
    ax.set_title(f"{title_prefix} Y-Coordinate Over Time - All Stimuli", fontsize=14, fontweight='bold')
    
    for i, y_kin in enumerate(y_kinematics):
        color = colors[i % len(colors)]
        ax.plot(y_kin['mean'], '-', linewidth=2.5, label=f'Stim {i}', color=color, alpha=0.9)
        ax.fill_between(range(len(y_kin['mean'])), y_kin['upper'], y_kin['lower'], 
                       color=color, alpha=0.15)
    
    if show_mid_point and len(y_kinematics[0]['mean']) > 100:
        mid_point = len(y_kinematics[0]['mean']) // 2
        ax.axvline(x=mid_point, color='gray', linestyle='--', alpha=0.5, linewidth=2, label='midpoint')
    
    # Add shaded region for extrema search range
    ax.axvspan(extrema_range_start, extrema_range_end, alpha=0.1, color='yellow', label='Extrema Search Range')
    
    # Add vertical lines for max velocity indices
    for i, max_vel_idx in enumerate(max_velocity_indices):
        color = colors[i % len(colors)]
        ax.axvline(x=max_vel_idx, color=color, linestyle=':', linewidth=2.5, alpha=0.8, 
                   label=f'Stim {i} Max Vel (idx={max_vel_idx})')
    
    ax.set_ylabel("Y Position", fontsize=12, fontweight='bold')
    ax.legend(loc='best', fontsize=10)
    ax.grid(alpha=0.3)
    
    # Z-coordinate overlay
    ax = axes[plot_idx]
    plot_idx += 1
    ax.set_title(f"{title_prefix} Z-Coordinate Over Time - All Stimuli", fontsize=14, fontweight='bold')
    
    for i, z_kin in enumerate(z_kinematics):
        color = colors[i % len(colors)]
        ax.plot(z_kin['mean'], '-', linewidth=2.5, label=f'Stim {i}', color=color, alpha=0.9)
        ax.fill_between(range(len(z_kin['mean'])), z_kin['upper'], z_kin['lower'], 
                       color=color, alpha=0.15)
    
    if show_mid_point and len(z_kinematics[0]['mean']) > 100:
        mid_point = len(z_kinematics[0]['mean']) // 2
        ax.axvline(x=mid_point, color='gray', linestyle='--', alpha=0.5, linewidth=2, label='midpoint')
    
    # Add shaded region for extrema search range
    ax.axvspan(extrema_range_start, extrema_range_end, alpha=0.1, color='yellow', label='Extrema Search Range')
    
    # Add vertical lines for max velocity indices
    for i, max_vel_idx in enumerate(max_velocity_indices):
        color = colors[i % len(colors)]
        ax.axvline(x=max_vel_idx, color=color, linestyle=':', linewidth=2.5, alpha=0.8, 
                   label=f'Stim {i} Max Vel (idx={max_vel_idx})')
    
    ax.set_ylabel("Z Position", fontsize=12, fontweight='bold')
    ax.legend(loc='best', fontsize=10)
    ax.grid(alpha=0.3)
    
    # Velocity overlay (use pre-computed velocity data)
    ax = axes[plot_idx]
    plot_idx += 1
    ax.set_title(f"{title_prefix} Velocity (3D Magnitude) Over Time - All Stimuli", fontsize=14, fontweight='bold')
    
    for i in range(n_stim):
        color = colors[i % len(colors)]
        ax.plot(velocity_data[i], '-', linewidth=2.5, label=f'Stim {i}', color=color, alpha=0.9)
        
        # Mark max velocity point
        max_vel_idx = max_velocity_indices[i]
        ax.plot(max_vel_idx, velocity_data[i][max_vel_idx], '*', markersize=15, color=color, 
               markeredgecolor='black', markeredgewidth=2, alpha=0.9)
    
    if show_mid_point and len(velocity_data[0]) > 100:
        mid_point = len(velocity_data[0]) // 2
        ax.axvline(x=mid_point, color='gray', linestyle='--', alpha=0.5, linewidth=2, label='midpoint')
    
    # Add shaded region for extrema search range
    ax.axvspan(extrema_range_start, extrema_range_end, alpha=0.1, color='yellow', label='Extrema Search Range')
    
    # Add vertical lines for max velocity indices
    for i, max_vel_idx in enumerate(max_velocity_indices):
        color = colors[i % len(colors)]
        ax.axvline(x=max_vel_idx, color=color, linestyle=':', linewidth=2.5, alpha=0.8, 
                   label=f'Stim {i} Max Vel (idx={max_vel_idx})')
    
    ax.set_ylabel("Velocity (3D)", fontsize=12, fontweight='bold')
    ax.legend(loc='best', fontsize=10)
    ax.grid(alpha=0.3)
    
    # Energy overlay with extrema marked within range
    ax = axes[plot_idx]
    plot_idx += 1
    ax.set_title(f"{title_prefix} Energy Over Time - All Stimuli (Extrema in Range [{extrema_range_start}-{extrema_range_end}])", fontsize=14, fontweight='bold')
    
    for i, eng in enumerate(energy):
        color = colors[i % len(colors)]
        energy_mean = np.array(eng['mean'])
        ax.plot(energy_mean, '-', linewidth=2.5, label=f'Stim {i}', color=color, alpha=0.9)
        ax.fill_between(range(len(eng['mean'])), eng['upper'], eng['lower'], 
                       color=color, alpha=0.15)
        
        # Find min and max within the specified range
        range_min_idx, range_max_idx = find_extrema_in_range(energy_mean, extrema_range_start, extrema_range_end)
        
        # Mark minimum within range
        ax.plot(range_min_idx, energy_mean[range_min_idx], 'v', markersize=12, color=color, 
               markeredgecolor='black', markeredgewidth=2, alpha=0.9, 
               label=f'Stim {i} Min (idx={range_min_idx})')
        
        # Mark maximum within range
        ax.plot(range_max_idx, energy_mean[range_max_idx], '^', markersize=12, color=color, 
               markeredgecolor='black', markeredgewidth=2, alpha=0.9,
               label=f'Stim {i} Max (idx={range_max_idx})')
    
    if show_mid_point and len(energy[0]['mean']) > 100:
        mid_point = len(energy[0]['mean']) // 2
        ax.axvline(x=mid_point, color='gray', linestyle='--', alpha=0.5, linewidth=2, label='midpoint')
    
    # Add shaded region for extrema search range
    ax.axvspan(extrema_range_start, extrema_range_end, alpha=0.1, color='yellow', label='Extrema Search Range')
    
    # Add vertical lines for max velocity indices
    for i, max_vel_idx in enumerate(max_velocity_indices):
        color = colors[i % len(colors)]
        ax.axvline(x=max_vel_idx, color=color, linestyle=':', linewidth=2.5, alpha=0.8, 
                   label=f'Stim {i} Max Vel (idx={max_vel_idx})')
    
    ax.set_ylabel("Energy", fontsize=12, fontweight='bold')
    ax.legend(loc='best', fontsize=9, ncol=2)
    ax.grid(alpha=0.3)
    
    # Boltzmann Probability overlay with extrema marked within range
    ax = axes[plot_idx]
    plot_idx += 1
    ax.set_title(f"{title_prefix} Boltzmann Probability Over Time - All Stimuli (Extrema in Range [{extrema_range_start}-{extrema_range_end}])", fontsize=14, fontweight='bold')
    
    for i, eng in enumerate(energy):
        color = colors[i % len(colors)]
        boltzmann_mean = np.exp(-np.array(eng['mean']) / temperature)
        boltzmann_upper = np.exp(-np.array(eng['lower']) / temperature)
        boltzmann_lower = np.exp(-np.array(eng['upper']) / temperature)
        
        ax.plot(boltzmann_mean, '-', linewidth=2.5, label=f'Stim {i}', color=color, alpha=0.9)
        ax.fill_between(range(len(boltzmann_mean)), boltzmann_upper, boltzmann_lower, 
                       color=color, alpha=0.15)
        
        # Find min and max within the specified range
        range_min_idx, range_max_idx = find_extrema_in_range(boltzmann_mean, extrema_range_start, extrema_range_end)
        
        # Mark minimum within range
        ax.plot(range_min_idx, boltzmann_mean[range_min_idx], 'v', markersize=12, color=color, 
               markeredgecolor='black', markeredgewidth=2, alpha=0.9,
               label=f'Stim {i} Min (idx={range_min_idx})')
        
        # Mark maximum within range
        ax.plot(range_max_idx, boltzmann_mean[range_max_idx], '^', markersize=12, color=color, 
               markeredgecolor='black', markeredgewidth=2, alpha=0.9,
               label=f'Stim {i} Max (idx={range_max_idx})')
    
    if show_mid_point and len(energy[0]['mean']) > 100:
        mid_point = len(energy[0]['mean']) // 2
        ax.axvline(x=mid_point, color='gray', linestyle='--', alpha=0.5, linewidth=2, label='midpoint')
    
    # Add shaded region for extrema search range
    ax.axvspan(extrema_range_start, extrema_range_end, alpha=0.1, color='yellow', label='Extrema Search Range')
    
    # Add vertical lines for max velocity indices
    for i, max_vel_idx in enumerate(max_velocity_indices):
        color = colors[i % len(colors)]
        ax.axvline(x=max_vel_idx, color=color, linestyle=':', linewidth=2.5, alpha=0.8, 
                   label=f'Stim {i} Max Vel (idx={max_vel_idx})')
    
    ax.set_ylabel("Boltzmann Probability", fontsize=12, fontweight='bold')
    ax.legend(loc='best', fontsize=9, ncol=2)
    ax.grid(alpha=0.3)
    
    # Firing Rate overlay with extrema marked within range (if available)
    if firing_rate is not None:
        ax = axes[plot_idx]
        plot_idx += 1
        ax.set_title(f"{title_prefix} Firing Rate Over Time - All Stimuli (Extrema in Range [{extrema_range_start}-{extrema_range_end}])", fontsize=14, fontweight='bold')
        
        for i, firing_data in enumerate(firing_rate):
            color = colors[i % len(colors)]
            firing_mean = np.array(firing_data['mean'])
            ax.plot(firing_mean, '-', linewidth=2.5, label=f'Stim {i}', color=color, alpha=0.9)
            ax.fill_between(range(len(firing_data['mean'])), 
                           firing_data['upper'],
                           firing_data['lower'],
                           color=color, alpha=0.15)
            
            # Find min and max within the specified range
            range_min_idx, range_max_idx = find_extrema_in_range(firing_mean, extrema_range_start, extrema_range_end)
            
            # Mark minimum within range
            ax.plot(range_min_idx, firing_mean[range_min_idx], 'v', markersize=12, color=color, 
                   markeredgecolor='black', markeredgewidth=2, alpha=0.9,
                   label=f'Stim {i} Min (idx={range_min_idx})')
            
            # Mark maximum within range
            ax.plot(range_max_idx, firing_mean[range_max_idx], '^', markersize=12, color=color, 
                   markeredgecolor='black', markeredgewidth=2, alpha=0.9,
                   label=f'Stim {i} Max (idx={range_max_idx})')
        
        if show_mid_point and len(firing_rate[0]['mean']) > 100:
            mid_point = len(firing_rate[0]['mean']) // 2
            ax.axvline(x=mid_point, color='gray', linestyle='--', alpha=0.5, linewidth=2, label='midpoint')
        
        # Add shaded region for extrema search range
        ax.axvspan(extrema_range_start, extrema_range_end, alpha=0.1, color='yellow', label='Extrema Search Range')
        
        # Add vertical lines for max velocity indices
        for i, max_vel_idx in enumerate(max_velocity_indices):
            color = colors[i % len(colors)]
            ax.axvline(x=max_vel_idx, color=color, linestyle=':', linewidth=2.5, alpha=0.8, 
                       label=f'Stim {i} Max Vel (idx={max_vel_idx})')
        
        ax.set_ylabel("Firing Rate", fontsize=12, fontweight='bold')
        ax.legend(loc='best', fontsize=9, ncol=2)
        ax.grid(alpha=0.3)
    
    # J values overlay (if available)
    if j_values is not None:
        ax = axes[plot_idx]
        plot_idx += 1
        ax.set_title(f"{title_prefix} J Values (Local Interactions) Over Time - All Stimuli", fontsize=14, fontweight='bold')
        
        for i, j_val in enumerate(j_values):
            color = colors[i % len(colors)]
            ax.plot(j_val['mean'], '-', linewidth=2.5, label=f'Stim {i}', color=color, alpha=0.9)
            ax.fill_between(range(len(j_val['mean'])), 
                           j_val['upper'],
                           j_val['lower'],
                           color=color, alpha=0.15)
        
        if show_mid_point and len(j_values[0]['mean']) > 100:
            mid_point = len(j_values[0]['mean']) // 2
            ax.axvline(x=mid_point, color='gray', linestyle='--', alpha=0.5, linewidth=2, label='midpoint')
        
        # Add shaded region for extrema search range
        ax.axvspan(extrema_range_start, extrema_range_end, alpha=0.1, color='yellow', label='Extrema Search Range')
        
        # Add vertical lines for max velocity indices
        for i, max_vel_idx in enumerate(max_velocity_indices):
            color = colors[i % len(colors)]
            ax.axvline(x=max_vel_idx, color=color, linestyle=':', linewidth=2.5, alpha=0.8, 
                       label=f'Stim {i} Max Vel (idx={max_vel_idx})')
        
        ax.set_ylabel("J Value", fontsize=12, fontweight='bold')
        ax.legend(loc='best', fontsize=10)
        ax.grid(alpha=0.3)
    
    # H values overlay (if available)
    if h_values is not None:
        ax = axes[plot_idx]
        plot_idx += 1
        ax.set_title(f"{title_prefix} H Values (Local Fields) Over Time - All Stimuli", fontsize=14, fontweight='bold')
        
        for i, h_val in enumerate(h_values):
            color = colors[i % len(colors)]
            ax.plot(h_val['mean'], '-', linewidth=2.5, label=f'Stim {i}', color=color, alpha=0.9)
            ax.fill_between(range(len(h_val['mean'])), 
                           h_val['upper'],
                           h_val['lower'],
                           color=color, alpha=0.15)
        
        if show_mid_point and len(h_values[0]['mean']) > 100:
            mid_point = len(h_values[0]['mean']) // 2
            ax.axvline(x=mid_point, color='gray', linestyle='--', alpha=0.5, linewidth=2, label='midpoint')
        
        # Add shaded region for extrema search range
        ax.axvspan(extrema_range_start, extrema_range_end, alpha=0.1, color='yellow', label='Extrema Search Range')
        
        # Add vertical lines for max velocity indices
        for i, max_vel_idx in enumerate(max_velocity_indices):
            color = colors[i % len(colors)]
            ax.axvline(x=max_vel_idx, color=color, linestyle=':', linewidth=2.5, alpha=0.8, 
                       label=f'Stim {i} Max Vel (idx={max_vel_idx})')
        
        ax.set_ylabel("H Value", fontsize=12, fontweight='bold')
        ax.legend(loc='best', fontsize=10)
        ax.grid(alpha=0.3)
    
    # Set x-label on the last plot
    axes[-1].set_xlabel("Time", fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"all_stimuli_overlayed.png"), dpi=300, bbox_inches='tight')
    plt.close()
    
    # Save combined data to CSV including extrema information
    max_length = max(len(x_kinematics[i]['mean']) for i in range(n_stim))
    combined_df = pd.DataFrame({'Time': range(max_length)})
    
    # Store extrema information (now within range)
    extrema_info = []
    
    # Add max velocity information for each stimulus
    for i in range(n_stim):
        max_vel_idx = max_velocity_indices[i]
        extrema_info.append({
            'Stimulus': i,
            'Metric': 'Velocity',
            'Range_Start': extrema_range_start,
            'Range_End': extrema_range_end,
            'Min_Index': None,  # Not tracking min velocity
            'Min_Value': None,
            'Max_Index': max_vel_idx,
            'Max_Value': velocity_data[i][max_vel_idx]
        })
    
    # Add all metrics for all stimuli
    for i in range(n_stim):
        if len(x_kinematics[i]['mean']) == max_length:
            combined_df[f'Stim_{i}_X_Mean'] = x_kinematics[i]['mean']
            combined_df[f'Stim_{i}_X_Upper'] = x_kinematics[i]['upper']
            combined_df[f'Stim_{i}_X_Lower'] = x_kinematics[i]['lower']
            
            combined_df[f'Stim_{i}_Y_Mean'] = y_kinematics[i]['mean']
            combined_df[f'Stim_{i}_Y_Upper'] = y_kinematics[i]['upper']
            combined_df[f'Stim_{i}_Y_Lower'] = y_kinematics[i]['lower']
            
            combined_df[f'Stim_{i}_Z_Mean'] = z_kinematics[i]['mean']
            combined_df[f'Stim_{i}_Z_Upper'] = z_kinematics[i]['upper']
            combined_df[f'Stim_{i}_Z_Lower'] = z_kinematics[i]['lower']
            
            # Add velocity
            combined_df[f'Stim_{i}_Velocity'] = velocity_data[i]
            
            combined_df[f'Stim_{i}_Energy_Mean'] = energy[i]['mean']
            combined_df[f'Stim_{i}_Energy_Upper'] = energy[i]['upper']
            combined_df[f'Stim_{i}_Energy_Lower'] = energy[i]['lower']
            
            # Record energy extrema within range
            energy_mean = np.array(energy[i]['mean'])
            energy_min_idx, energy_max_idx = find_extrema_in_range(energy_mean, extrema_range_start, extrema_range_end)
            extrema_info.append({
                'Stimulus': i,
                'Metric': 'Energy',
                'Range_Start': extrema_range_start,
                'Range_End': extrema_range_end,
                'Min_Index': energy_min_idx,
                'Min_Value': energy_mean[energy_min_idx],
                'Max_Index': energy_max_idx,
                'Max_Value': energy_mean[energy_max_idx]
            })
            
            boltzmann_mean = np.exp(-np.array(energy[i]['mean']) / temperature)
            boltzmann_upper = np.exp(-np.array(energy[i]['lower']) / temperature)
            boltzmann_lower = np.exp(-np.array(energy[i]['upper']) / temperature)
            combined_df[f'Stim_{i}_Boltzmann_Mean'] = boltzmann_mean
            combined_df[f'Stim_{i}_Boltzmann_Upper'] = boltzmann_upper
            combined_df[f'Stim_{i}_Boltzmann_Lower'] = boltzmann_lower
            
            # Record Boltzmann extrema within range
            boltz_min_idx, boltz_max_idx = find_extrema_in_range(boltzmann_mean, extrema_range_start, extrema_range_end)
            extrema_info.append({
                'Stimulus': i,
                'Metric': 'Boltzmann',
                'Range_Start': extrema_range_start,
                'Range_End': extrema_range_end,
                'Min_Index': boltz_min_idx,
                'Min_Value': boltzmann_mean[boltz_min_idx],
                'Max_Index': boltz_max_idx,
                'Max_Value': boltzmann_mean[boltz_max_idx]
            })
            
            if firing_rate is not None:
                combined_df[f'Stim_{i}_FiringRate_Mean'] = firing_rate[i]['mean']
                combined_df[f'Stim_{i}_FiringRate_Upper'] = firing_rate[i]['upper']
                combined_df[f'Stim_{i}_FiringRate_Lower'] = firing_rate[i]['lower']
                
                # Record firing rate extrema within range
                firing_mean = np.array(firing_rate[i]['mean'])
                firing_min_idx, firing_max_idx = find_extrema_in_range(firing_mean, extrema_range_start, extrema_range_end)
                extrema_info.append({
                    'Stimulus': i,
                    'Metric': 'FiringRate',
                    'Range_Start': extrema_range_start,
                    'Range_End': extrema_range_end,
                    'Min_Index': firing_min_idx,
                    'Min_Value': firing_mean[firing_min_idx],
                    'Max_Index': firing_max_idx,
                    'Max_Value': firing_mean[firing_max_idx]
                })
            
            if j_values is not None:
                combined_df[f'Stim_{i}_J_Mean'] = j_values[i]['mean']
                combined_df[f'Stim_{i}_J_Upper'] = j_values[i]['upper']
                combined_df[f'Stim_{i}_J_Lower'] = j_values[i]['lower']
            
            if h_values is not None:
                combined_df[f'Stim_{i}_H_Mean'] = h_values[i]['mean']
                combined_df[f'Stim_{i}_H_Upper'] = h_values[i]['upper']
                combined_df[f'Stim_{i}_H_Lower'] = h_values[i]['lower']
    
    combined_df.to_csv(os.path.join(output_dir, f"all_stimuli_overlayed_data.csv"), index=False)
    
    # Save extrema information to separate CSV
    extrema_df = pd.DataFrame(extrema_info)
    extrema_df.to_csv(os.path.join(output_dir, f"extrema_summary.csv"), index=False)
    
    print(f"Overlayed plot for all stimuli saved to {output_dir}")
    print(f"Extrema summary (within range [{extrema_range_start}-{extrema_range_end}]) saved to {output_dir}/extrema_summary.csv")

In [32]:
all_reach_states

['/data001/projects/ising/energy_decomposition/210423_results/210423_rep1/begin_reach/per_reach_state.csv',
 '/data001/projects/ising/energy_decomposition/210423_results/210423_rep1/mid_reach/per_reach_state.csv',
 '/data001/projects/ising/energy_decomposition/210423_results/210423_rep1/post_reach/per_reach_state.csv',
 '/data001/projects/ising/energy_decomposition/210423_results/210423_rep1/full_reach/per_reach_state.csv',
 '/data001/projects/ising/energy_decomposition/210425_results/210425_rep1/begin_reach/per_reach_state.csv',
 '/data001/projects/ising/energy_decomposition/210425_results/210425_rep1/mid_reach/per_reach_state.csv',
 '/data001/projects/ising/energy_decomposition/210425_results/210425_rep1/post_reach/per_reach_state.csv',
 '/data001/projects/ising/energy_decomposition/210425_results/210425_rep1/full_reach/per_reach_state.csv',
 '/data001/projects/ising/energy_decomposition/210511_results/210511_rep1/begin_reach/per_reach_state.csv',
 '/data001/projects/ising/energy_dec

In [35]:
all_reach_states[0].split('_results')[0]

'/data001/projects/ising/energy_decomposition/210423'

In [39]:
import os
import multiprocessing as mp
matlab_file = "/data001/projects/enserrog/AbigailData/energy_over_time/"

pool = mp.Pool(processes=8)

def _generate_results_of_session(session):
    if not "full_reach" in session:
        return
        
    print(session)
    session_id = session.split('_results')[0][-6:]
    print(f"Session_id: {session_id}")
    
    df = pd.read_csv(session)
    
    # Calculate statistics
    stats = calculate_statistics_from_dataframe(df, confidence=0.8, output_dir=f'./stats_output_4/{session_id}')
    
    # Plot the results
    plot_all_overlayed(stats, critical_energy=0.5, output_dir=f'./plots_output_4/{session_id}')


pool.map(_generate_results_of_session, all_reach_states)

/data001/projects/ising/energy_decomposition/210511_results/210511_rep1/full_reach/per_reach_state.csv/data001/projects/ising/energy_decomposition/210423_results/210423_rep1/full_reach/per_reach_state.csv/data001/projects/ising/energy_decomposition/210425_results/210425_rep1/full_reach/per_reach_state.csv/data001/projects/ising/energy_decomposition/210512_results/210512_rep1/full_reach/per_reach_state.csv/data001/projects/ising/energy_decomposition/210514_results/210514_rep1/full_reach/per_reach_state.csv
/data001/projects/ising/energy_decomposition/210606_results/210606_rep1/full_reach/per_reach_state.csv/data001/projects/ising/energy_decomposition/210614_results/210614_rep1/full_reach/per_reach_state.csv
/data001/projects/ising/energy_decomposition/210515_results/210515_rep1/full_reach/per_reach_state.csv




Session_id: 210423Session_id: 210425Session_id: 210512Session_id: 210614Session_id: 210515Session_id: 210511Session_id: 210606







Session_id: 210514
Finding extrema within r

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]